# 修订记录与复现注意事项

本版本修正：城市/日期空值及重复键检查；仅对明确的外生数值协变量进行城市内前向填充；AQI目标和污染物观测不插补；增加插补前后缺失审计；Winsorization训练样本不足时跳过而不使用全时段统计量；FAR统一为FP/(FP+TN)；强制滚动因果CEEMDAN；删除重复的可视化代码单元。

**仍需真实数据运行与论文同步核验：** 当前 `fit_predict_ensemble` 将 `ENSEMBLE_SEEDS` 的预测取均值，因此不能将其结果称为单随机种子的单模型预测。Ridge融合和偏差校正使用验证集标签拟合，验证集分数属于拟合内诊断，不是独立验证性能。Brier Score/ECE 尚未实现经验证集拟合、测试集独立评估的概率校准器，当前应保持缺失，不得宣称已完成校准。城市/省级结果、混淆矩阵、消融日志和Bootstrap仍需实际运行核验。


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import math
import copy
import json
import random
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import radians, sin, cos, asin, sqrt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, average_precision_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import optuna
from PyEMD.CEEMDAN import CEEMDAN
from PyEMD.EMD import EMD
BASE_SEED = 42
SEARCH_SEED = 42
ENSEMBLE_SEEDS = [42, 52, 62]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
DATA_PATH_CANDIDATES = [
    "./data/input_dataset.xlsx"
]
DATA_PATH = None
for p in DATA_PATH_CANDIDATES:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError(f'未找到数据文件，请检查路径。候选路径：{DATA_PATH_CANDIDATES}')
SHEET_NAME = 'Sheet1'
SAVE_DIR = "./outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
history_len = 14
pred_horizon = 1
train_ratio = 0.7
val_ratio = 0.15
batch_size = 32
max_search_epochs = 35
search_patience = 7
final_max_epochs_cap = 60
MIN_FINAL_EPOCH = 6
OPTUNA_TRIALS = 18
OPTUNA_SPLITS = 4
USE_WINSORIZE = False
EXTREME_CLIP_Q_LOW = 0.01
EXTREME_CLIP_Q_HIGH = 0.99
MAX_CEEMDAN_TRIALS = 10
CAUSAL_CEEMDAN = True
CEEMDAN_NOISE_WIDTH = 0.05
MAX_IMF_KEEP = 6
USE_RESIDUAL_COMPONENT = True
STATIC_K = 3
STATIC_SELF_LOOP_WEIGHT = 1.0
STATIC_GRAPH_SYMMETRIC = True
DROP_HIGH_CORR = True
CORR_THRESHOLD = 0.98
MIN_STD = 1e-08
MRMR_TOP_PRESELECT = 24
STABILITY_BOOTSTRAPS = 50
STABILITY_BLOCK_RATIO = 0.75
STABILITY_THRESHOLD = 0.6
FINAL_TOP_K_MAX = 20
TARGET_COL = 'AQI'
DATE_COL = '日期'
CITY_COL = 'city'
POLLUTANT_COLS = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3_8h']
MANDATORY_FEATURES = ['AQI', 'AQI_lag1', 'PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3_8h']
STABILITY_XGB_PARAMS = dict(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, random_state=BASE_SEED, n_jobs=-1, verbosity=0, objective='reg:squarederror')
EVENT_THRESHOLDS = [100, 150, 200]
LOSS_W_MSE = 0.4
LOSS_W_HUBER = 0.25
LOSS_W_SMAPE = 0.15
LOSS_W_EVENT = 0.2
PEAK_Q = 0.85
PEAK_WEIGHT = 2.5
HUBER_DELTA = 1.0
EVENT_FOCUS_THR = 100.0
AQI_MIN = 0.0
AQI_MAX = 500.0
PROVINCE_WEIGHT_METHOD = 'raw_graph_node_strength'
FUSION_ALPHAS = [0.01, 0.1, 1.0, 10.0]
CITY_COORDS = {'长春': (125.324501, 43.886841), '吉林': (126.550727, 43.84559), '四平': (124.350398, 43.175755), '辽源': (125.143755, 42.887394), '通化': (125.936501, 41.728401), '白山': (126.424317, 41.942505), '松原': (124.823608, 45.136049), '白城': (122.836498, 45.618713), '延边': (129.517646, 42.904543)}

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_global_seed(BASE_SEED)

def ensure_dir(path):
    if path is None or path == '':
        return
    os.makedirs(path, exist_ok=True)

def safe_filename(name: str) -> str:
    invalid_chars = '\\\\/:*?"<>|'
    for ch in invalid_chars:
        name = name.replace(ch, '_')
    return str(name)

def calc_metrics(y_true, y_pred, eps=1e-08):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100.0
    r2 = r2_score(y_true, y_pred)
    return {'MSE': float(mse), 'RMSE': float(rmse), 'MAE': float(mae), 'MAPE': float(mape), 'R2': float(r2)}

def fit_robust_scaler(arr, eps=1e-06):
    arr = np.asarray(arr, dtype=np.float32)
    center = np.median(arr, axis=0, keepdims=True).astype(np.float32)
    q1 = np.percentile(arr, 25, axis=0, keepdims=True).astype(np.float32)
    q3 = np.percentile(arr, 75, axis=0, keepdims=True).astype(np.float32)
    scale = np.maximum(q3 - q1, eps).astype(np.float32)
    return (center, scale)

def transform_with_scaler(arr, center, scale):
    return ((arr - center) / scale).astype(np.float32)

def inverse_transform(arr_std, center, scale):
    return (arr_std * scale + center).astype(np.float32)

def save_curve(train_loss_list, val_loss_list, title, save_path):
    ensure_dir(os.path.dirname(save_path))
    plt.figure(figsize=(8, 5))
    plt.plot(train_loss_list, label='Train')
    plt.plot(val_loss_list, label='Val')
    plt.legend()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=220)
    plt.close()

def winsorize_by_city_train_quantiles(df_in, cols, train_dates, city_col='city', date_col='日期', q_low=0.01, q_high=0.99):
    df_out = df_in.copy()
    stats_rows = []
    train_mask_global = df_out[date_col].isin(train_dates)
    for city in sorted(df_out[city_col].unique()):
        city_mask = df_out[city_col] == city
        city_train_mask = city_mask & train_mask_global
        for col in cols:
            if col not in df_out.columns:
                continue
            series_train = pd.to_numeric(df_out.loc[city_train_mask, col], errors='coerce').dropna()
            if len(series_train) < 10:
                stats_rows.append({'City': city, 'Column': col, 'Lower_Train_Q': np.nan, 'Upper_Train_Q': np.nan, 'Clipped_Low_Count': 0, 'Clipped_High_Count': 0, 'Status': 'Skipped: fewer than 10 non-missing training values'})
                continue
            lower = float(series_train.quantile(q_low))
            upper = float(series_train.quantile(q_high))
            before_vals = pd.to_numeric(df_out.loc[city_mask, col], errors='coerce')
            low_cnt = int((before_vals < lower).sum())
            high_cnt = int((before_vals > upper).sum())
            df_out.loc[city_mask, col] = before_vals.clip(lower, upper)
            stats_rows.append({'City': city, 'Column': col, 'Lower_Train_Q': lower, 'Upper_Train_Q': upper, 'Clipped_Low_Count': low_cnt, 'Clipped_High_Count': high_cnt})
    return (df_out, pd.DataFrame(stats_rows))

def zero_crossing_count(x):
    s = np.sign(x).copy()
    s[s == 0] = 1
    return int(np.sum(s[1:] * s[:-1] < 0))

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = (lon2 - lon1, lat2 - lat1)
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * asin(sqrt(a)) * 6371.0
AQI_BP = np.array([0, 50, 100, 150, 200, 300, 400, 500], dtype=np.float32)
POLLUTANT_BREAKPOINTS = {'PM2.5': {'bp': np.array([0, 35, 75, 115, 150, 250, 350, 500], dtype=np.float32), 'iaqi': AQI_BP.copy()}, 'PM10': {'bp': np.array([0, 50, 150, 250, 350, 420, 500, 600], dtype=np.float32), 'iaqi': AQI_BP.copy()}, 'SO2': {'bp': np.array([0, 50, 150, 475, 800, 1600, 2100, 2620], dtype=np.float32), 'iaqi': AQI_BP.copy()}, 'NO2': {'bp': np.array([0, 40, 80, 180, 280, 565, 750, 940], dtype=np.float32), 'iaqi': AQI_BP.copy()}, 'CO': {'bp': np.array([0, 2, 4, 14, 24, 36, 48, 60], dtype=np.float32), 'iaqi': AQI_BP.copy()}, 'O3_8h': {'bp': np.array([0, 100, 160, 215, 265, 800], dtype=np.float32), 'iaqi': np.array([0, 50, 100, 150, 200, 300], dtype=np.float32)}}

def _piecewise_iaqi_np(cp, bp, iaqi):
    cp = np.maximum(cp, 0.0)
    out = np.zeros_like(cp, dtype=np.float32)
    for i in range(len(bp) - 1):
        lo, hi = (bp[i], bp[i + 1])
        ia_lo, ia_hi = (iaqi[i], iaqi[i + 1])
        if i == len(bp) - 2:
            mask = (cp >= lo) & (cp <= hi)
        else:
            mask = (cp >= lo) & (cp < hi)
        out[mask] = (ia_hi - ia_lo) / (hi - lo + 1e-08) * (cp[mask] - lo) + ia_lo
    out[cp > bp[-1]] = iaqi[-1]
    return out.astype(np.float32)

def china_aqi_from_pollutants_np(poll_true):
    iaqi_list = []
    for j, p in enumerate(POLLUTANT_COLS):
        rule = POLLUTANT_BREAKPOINTS[p]
        iaqi = _piecewise_iaqi_np(poll_true[..., j], rule['bp'], rule['iaqi'])
        iaqi_list.append(iaqi)
    iaqi_stack = np.stack(iaqi_list, axis=-1)
    aqi_rebuild = iaqi_stack.max(axis=-1)
    primary_idx = iaqi_stack.argmax(axis=-1)
    return (aqi_rebuild.astype(np.float32), primary_idx.astype(np.int64), iaqi_stack.astype(np.float32))

def aqi_grade_np(aqi):
    aqi = np.asarray(aqi)
    bins = np.array([0, 50, 100, 150, 200, 300, 500], dtype=np.float32)
    return np.digitize(aqi, bins[1:], right=True)

def expected_calibration_error(y_true_bin, prob, n_bins=10):
    y_true_bin = np.asarray(y_true_bin).reshape(-1)
    prob = np.asarray(prob).reshape(-1)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(prob)
    for i in range(n_bins):
        if i < n_bins - 1:
            mask = (prob >= bins[i]) & (prob < bins[i + 1])
        else:
            mask = (prob >= bins[i]) & (prob <= bins[i + 1])
        if mask.sum() == 0:
            continue
        acc = y_true_bin[mask].mean()
        conf = prob[mask].mean()
        ece += mask.sum() / n * abs(acc - conf)
    return float(ece)

def event_prob_from_aqi(y_pred, thr, temp=18.0):
    y_pred = np.asarray(y_pred).reshape(-1)
    prob = 1.0 / (1.0 + np.exp(-(y_pred - thr) / temp))
    return np.clip(prob, 1e-06, 1 - 1e-06)

def calc_multithreshold_event_metrics(y_true, y_pred, thresholds=(100, 150, 200), eps=1e-08):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError('事件评估包含非有限值')
    out = {}
    for thr in thresholds:
        truth = y_true >= thr
        forecast = y_pred >= thr
        tp = int(np.sum(truth & forecast))
        fp = int(np.sum(~truth & forecast))
        fn = int(np.sum(truth & ~forecast))
        tn = int(np.sum(~truth & ~forecast))
        n = tp + fp + fn + tn
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
        far = fp / (fp + tn) if fp + tn else np.nan
        csi = tp / (tp + fp + fn) if tp + fp + fn else np.nan
        auc = roc_auc_score(truth.astype(int), y_pred) if truth.any() and (~truth).any() else np.nan
        prauc = average_precision_score(truth.astype(int), y_pred) if truth.any() and (~truth).any() else np.nan
        values = dict(TP=tp, FP=fp, FN=fn, TN=tn, Accuracy=(tp + tn) / n if n else np.nan, EHR=recall, Recall=recall, Precision=precision, F1=f1, FAR=far, CSI=csi, AUC=auc, PRAUC=prauc)
        for k, v in values.items():
            out[f'{k}_{thr}'] = float(v) if not isinstance(v, int) else v
        out[f'Brier_{thr}'] = np.nan
        out[f'ECE_{thr}'] = np.nan
    return out

def calc_extended_metrics(y_true, y_pred, thresholds=(100, 150, 200), eps=1e-08):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    base = calc_metrics(y_true, y_pred, eps=eps)
    mbe = np.mean(y_pred - y_true)
    smape = np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100.0
    multi_thr = calc_multithreshold_event_metrics(y_true, y_pred, thresholds=thresholds, eps=eps)
    return {**base, 'MBE': float(mbe), 'SMAPE': float(smape), **multi_thr}
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
required_cols = [DATE_COL, CITY_COL, TARGET_COL] + POLLUTANT_COLS
missing_required = [c for c in required_cols if c not in df_raw.columns]
if len(missing_required) > 0:
    raise ValueError(f'数据缺少必要列: {missing_required}')
if df_raw[DATE_COL].isna().any():
    raise ValueError('日期字段存在空值')
if df_raw[CITY_COL].isna().any():
    raise ValueError('城市字段存在空值')
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors='coerce')
if df_raw[DATE_COL].isna().any():
    raise ValueError('日期字段存在无法解析的值')
df_raw[CITY_COL] = df_raw[CITY_COL].astype(str).str.strip()
if df_raw[CITY_COL].eq('').any():
    raise ValueError('城市字段存在空字符串')
if df_raw.duplicated([CITY_COL, DATE_COL]).any():
    dup = df_raw.loc[df_raw.duplicated([CITY_COL, DATE_COL], keep=False), [CITY_COL, DATE_COL]]
    raise ValueError(f'存在重复的城市-日期记录，示例：\\n{dup.head(10)}')
df_raw = df_raw.sort_values([CITY_COL, DATE_COL], kind='stable').reset_index(drop=True)
city_order = sorted(df_raw[CITY_COL].unique().tolist())
missing_coords = [c for c in city_order if c not in CITY_COORDS]
if missing_coords:
    raise ValueError(f'以下城市缺少坐标信息: {missing_coords}')
print('原始数据形状:', df_raw.shape)
print('城市:', city_order)
print('日期范围:', df_raw[DATE_COL].min(), '->', df_raw[DATE_COL].max())
raw_numeric_cols = [c for c in df_raw.columns if c not in [DATE_COL, CITY_COL] and pd.api.types.is_numeric_dtype(df_raw[c])]
print('原始数值列数量:', len(raw_numeric_cols))
dates_for_clip = sorted(df_raw[DATE_COL].dropna().unique())
train_end_for_clip = int(len(dates_for_clip) * train_ratio)
train_dates_for_clip = set(dates_for_clip[:train_end_for_clip])
if USE_WINSORIZE:
    clip_cols = [c for c in ['AQI'] + POLLUTANT_COLS if c in df_raw.columns]
    df_raw, extreme_clip_stats = winsorize_by_city_train_quantiles(df_in=df_raw, cols=clip_cols, train_dates=train_dates_for_clip, q_low=EXTREME_CLIP_Q_LOW, q_high=EXTREME_CLIP_Q_HIGH)
else:
    extreme_clip_stats = pd.DataFrame(columns=['City', 'Column', 'Lower_Train_Q', 'Upper_Train_Q', 'Clipped_Low_Count', 'Clipped_High_Count'])
extreme_clip_stats.to_excel(os.path.join(SAVE_DIR, 'extreme_value_clip_summary.xlsx'), index=False)
num_cols_raw = [c for c in df_raw.columns if c not in [DATE_COL, CITY_COL] and pd.api.types.is_numeric_dtype(df_raw[c])]
missing_before_rows = []
for city, group in df_raw.groupby(CITY_COL, sort=False):
    for col in num_cols_raw:
        missing_before_rows.append({'City': city, 'Feature': col, 'Missing_Count': int(group[col].isna().sum()), 'Total_Count': int(len(group)), 'Missing_Rate': float(group[col].isna().mean()) if len(group) else 0.0})
missing_before_df = pd.DataFrame(missing_before_rows)
missing_before_df.to_excel(os.path.join(SAVE_DIR, 'missingness_before_fill.xlsx'), index=False)
PROTECTED_OBSERVATION_COLS = {TARGET_COL, *POLLUTANT_COLS}
covariate_fill_cols = [c for c in num_cols_raw if c not in PROTECTED_OBSERVATION_COLS]
protected_before = df_raw[list(PROTECTED_OBSERVATION_COLS)].copy()
if covariate_fill_cols:
    df_raw.loc[:, covariate_fill_cols] = df_raw.groupby(CITY_COL, sort=False)[covariate_fill_cols].ffill()
for col in PROTECTED_OBSERVATION_COLS:
    if not df_raw[col].equals(protected_before[col]):
        raise AssertionError(f'受保护的真实观测列被插补或修改：{col}')
missing_after_rows = []
for city, group in df_raw.groupby(CITY_COL, sort=False):
    for col in num_cols_raw:
        missing_after_rows.append({'City': city, 'Feature': col, 'Missing_Count': int(group[col].isna().sum()), 'Total_Count': int(len(group)), 'Missing_Rate': float(group[col].isna().mean()) if len(group) else 0.0})
missing_after_df = pd.DataFrame(missing_after_rows)
missing_after_df.to_excel(os.path.join(SAVE_DIR, 'missingness_after_fill.xlsx'), index=False)
pd.DataFrame([{'Stage': 'Before covariate forward-fill', 'Missing_Cells': int(missing_before_df['Missing_Count'].sum()), 'Audited_Cells': int(missing_before_df['Total_Count'].sum()), 'Missing_Rate_Percent': 100.0 * missing_before_df['Missing_Count'].sum() / max(1, int(missing_before_df['Total_Count'].sum()))}, {'Stage': 'After covariate forward-fill', 'Missing_Cells': int(missing_after_df['Missing_Count'].sum()), 'Audited_Cells': int(missing_after_df['Total_Count'].sum()), 'Missing_Rate_Percent': 100.0 * missing_after_df['Missing_Count'].sum() / max(1, int(missing_after_df['Total_Count'].sum()))}]).to_excel(os.path.join(SAVE_DIR, 'missingness_imputation_summary.xlsx'), index=False)
df_fs = df_raw.copy()
df_fs['AQI_lag1'] = df_fs.groupby(CITY_COL)['AQI'].shift(1)
candidate_pool = raw_numeric_cols.copy()
if 'AQI_lag1' not in candidate_pool:
    candidate_pool.append('AQI_lag1')
target_future_col = f'{TARGET_COL}_tplus{pred_horizon}'
df_fs[target_future_col] = df_fs.groupby(CITY_COL)[TARGET_COL].shift(-pred_horizon)
fs_use_cols = [DATE_COL, CITY_COL, target_future_col] + candidate_pool
df_fs = df_fs[fs_use_cols].copy()
df_fs = df_fs.dropna(subset=[target_future_col] + candidate_pool).reset_index(drop=True)
if df_fs.empty:
    raise ValueError('特征选择数据为空，请检查 AQI_lag1 与未来目标构造。')
print('候选特征数（原始特征 + AQI_lag1）:', len(candidate_pool))
print('特征选择可用样本数:', df_fs.shape[0])
all_fs_dates = sorted(df_raw[DATE_COL].dropna().unique())
train_cutoff_fs = all_fs_dates[int(len(all_fs_dates) * train_ratio)]
val_cutoff_fs = all_fs_dates[int(len(all_fs_dates) * (train_ratio + val_ratio))]
calendar_map = {d: k for k, d in enumerate(all_fs_dates)}

def _target_date_from_origin(d):
    pos = calendar_map.get(d)
    return all_fs_dates[pos + pred_horizon] if pos is not None and pos + pred_horizon < len(all_fs_dates) else pd.NaT
df_fs['__target_date'] = df_fs[DATE_COL].map(_target_date_from_origin)
train_df_fs = df_fs[df_fs['__target_date'] < train_cutoff_fs].copy()
val_df_fs = df_fs[(df_fs['__target_date'] >= train_cutoff_fs) & (df_fs['__target_date'] < val_cutoff_fs)].copy()
test_df_fs = df_fs[df_fs['__target_date'] >= val_cutoff_fs].copy()
candidate_features = candidate_pool.copy()
std_series = train_df_fs[candidate_features].std(numeric_only=True)
non_constant_features = std_series[std_series > MIN_STD].index.tolist()
dropped_constant = sorted(list(set(candidate_features) - set(non_constant_features)))
candidate_features = non_constant_features
dropped_high_corr = []
if DROP_HIGH_CORR and len(candidate_features) > 1:
    corr_mat = train_df_fs[candidate_features].corr().abs()
    upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > CORR_THRESHOLD)]
    dropped_high_corr = sorted(to_drop)
    candidate_features = [c for c in candidate_features if c not in to_drop]
print('去常数后特征数:', len(non_constant_features))
print('去高相关后特征数:', len(candidate_features))

def time_safe_mrmr_select(X: pd.DataFrame, y: pd.Series, top_k: int) -> Tuple[List[str], pd.DataFrame]:
    feature_names = X.columns.tolist()
    relevance = mutual_info_regression(X.values, y.values, random_state=BASE_SEED)
    relevance = pd.Series(relevance, index=feature_names).fillna(0.0)
    corr = X.corr().abs().fillna(0.0)
    selected = []
    records = []
    for step in range(min(top_k, len(feature_names))):
        if step == 0:
            best_feat = relevance.sort_values(ascending=False).index[0]
            best_score = relevance.loc[best_feat]
            avg_red = 0.0
        else:
            remain = [f for f in feature_names if f not in selected]
            best_feat = None
            best_score = -1e+18
            avg_red = None
            for f in remain:
                red = corr.loc[f, selected].mean() if len(selected) > 0 else 0.0
                score = relevance.loc[f] - red
                if score > best_score:
                    best_score = score
                    best_feat = f
                    avg_red = red
        selected.append(best_feat)
        records.append({'Step': step + 1, 'Feature': best_feat, 'Relevance_MI': float(relevance.loc[best_feat]), 'Avg_Redundancy': float(avg_red), 'mRMR_Score': float(best_score)})
    return (selected, pd.DataFrame(records))

def block_subsample_dates(date_array, ratio, rng):
    date_array = np.array(sorted(date_array))
    n = len(date_array)
    block_len = max(8, int(np.ceil(n * ratio)))
    block_len = min(block_len, n)
    start = rng.randint(0, n - block_len + 1)
    return date_array[start:start + block_len]
X_train_fs = train_df_fs[candidate_features].copy()
y_train_fs = train_df_fs[target_future_col].copy()
mrmr_features, mrmr_rank_df = time_safe_mrmr_select(X=X_train_fs, y=y_train_fs, top_k=min(MRMR_TOP_PRESELECT, len(candidate_features)))
mrmr_rank_df.to_excel(os.path.join(SAVE_DIR, 'mrmr_ranking.xlsx'), index=False)
print('mRMR 预筛特征数:', len(mrmr_features))
print('mRMR 预筛特征:', mrmr_features)
rng = np.random.RandomState(BASE_SEED)
stability_counts = pd.Series(0, index=mrmr_features, dtype=np.int32)
train_unique_dates_inner = np.array(sorted(train_df_fs[DATE_COL].unique()))
for _ in range(STABILITY_BOOTSTRAPS):
    sampled_dates = block_subsample_dates(train_unique_dates_inner, STABILITY_BLOCK_RATIO, rng)
    boot_df = train_df_fs[train_df_fs[DATE_COL].isin(sampled_dates)].copy()
    if len(boot_df) < 50:
        continue
    X_boot = boot_df[mrmr_features]
    y_boot = boot_df[target_future_col]
    model = XGBRegressor(**STABILITY_XGB_PARAMS)
    model.fit(X_boot, y_boot)
    imp = pd.Series(model.feature_importances_, index=mrmr_features)
    imp = imp.sort_values(ascending=False)
    keep_k = max(5, int(np.ceil(len(mrmr_features) * 0.4)))
    chosen = imp.head(keep_k).index.tolist()
    stability_counts.loc[chosen] += 1
stability_df = pd.DataFrame({'Feature': mrmr_features, 'Stability_Count': stability_counts.values, 'Stability_Freq': stability_counts.values / max(1, STABILITY_BOOTSTRAPS)}).sort_values(['Stability_Freq', 'Stability_Count'], ascending=False).reset_index(drop=True)
stable_features = stability_df.loc[stability_df['Stability_Freq'] >= STABILITY_THRESHOLD, 'Feature'].tolist()
if len(stable_features) == 0:
    stable_features = stability_df.head(min(10, len(stability_df)))['Feature'].tolist()
mandatory_existing = [f for f in MANDATORY_FEATURES if f in candidate_features]
selected_features = list(dict.fromkeys(stable_features + mandatory_existing))
if len(selected_features) > FINAL_TOP_K_MAX:
    stable_rank = stability_df['Feature'].tolist()
    ordered = []
    for f in mandatory_existing:
        if f not in ordered:
            ordered.append(f)
    for f in stable_rank:
        if f not in ordered:
            ordered.append(f)
    selected_features = ordered[:FINAL_TOP_K_MAX]
selection_source = []
for f in selected_features:
    source = []
    if f in stable_features:
        source.append('stable')
    if f in mandatory_existing:
        source.append('mandatory')
    selection_source.append({'Feature': f, 'Source': '+'.join(source)})
selection_source_df = pd.DataFrame(selection_source)
stability_df['Selected'] = stability_df['Feature'].isin(selected_features)
stability_df.to_excel(os.path.join(SAVE_DIR, 'stability_selection.xlsx'), index=False)
selection_source_df.to_excel(os.path.join(SAVE_DIR, 'selected_feature_sources.xlsx'), index=False)
print('最终选中特征数:', len(selected_features))
print('最终选中特征:', selected_features)
final_fs_model = XGBRegressor(**STABILITY_XGB_PARAMS)
final_fs_model.fit(train_df_fs[selected_features], train_df_fs[target_future_col])
pred_val_fs = final_fs_model.predict(val_df_fs[selected_features])
pred_test_fs = final_fs_model.predict(test_df_fs[selected_features])
surrogate_metrics_df = pd.DataFrame([{'Phase': 'Val', **calc_metrics(val_df_fs[target_future_col], pred_val_fs)}, {'Phase': 'Test', **calc_metrics(test_df_fs[target_future_col], pred_test_fs)}])
surrogate_metrics_df.to_excel(os.path.join(SAVE_DIR, 'surrogate_model_metrics.xlsx'), index=False)
pd.DataFrame({'Selected_Features': selected_features}).to_excel(os.path.join(SAVE_DIR, 'selected_feature_list.xlsx'), index=False)
df_model = df_raw.copy()
if 'AQI_lag1' not in df_model.columns:
    df_model['AQI_lag1'] = df_model.groupby(CITY_COL)['AQI'].shift(1)
keep_cols_for_model = list(dict.fromkeys([DATE_COL, CITY_COL, TARGET_COL] + POLLUTANT_COLS + selected_features + ['AQI_lag1']))
keep_cols_existing = [c for c in keep_cols_for_model if c in df_model.columns]
df_model = df_model[keep_cols_existing].copy()
required_non_na_cols = list(dict.fromkeys([TARGET_COL] + POLLUTANT_COLS + selected_features))
df_model = df_model.dropna(subset=required_non_na_cols).reset_index(drop=True)
complete_dates = []
for dt, grp in df_model.groupby(DATE_COL):
    cities_here = set(grp[CITY_COL].tolist())
    if len(cities_here) == len(city_order):
        complete_dates.append(dt)
complete_dates = sorted(complete_dates)
df_model = df_model[df_model[DATE_COL].isin(complete_dates)].copy().reset_index(drop=True)
if len(complete_dates) == 0:
    raise ValueError('拼接后的建模日期为空，请检查特征选择结果与数据完整性。')
print('建模完整日期数:', len(complete_dates))
print('建模样本数:', df_model.shape[0])
dates = complete_dates
T_all = len(dates)
train_end = int(T_all * train_ratio)
val_end = int(T_all * (train_ratio + val_ratio))
train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]
print(f'主模型 Total dates={T_all}, train={len(train_dates)}, val={len(val_dates)}, test={len(test_dates)}')

def build_pivot(df_in, value_col, city_order):
    p = df_in.pivot(index=DATE_COL, columns=CITY_COL, values=value_col).sort_index()
    p = p.reindex(columns=city_order)
    return p
pivot_aqi = build_pivot(df_model, TARGET_COL, city_order).loc[dates]
pollutant_pivots = {}
for p in POLLUTANT_COLS:
    pollutant_pivots[p] = build_pivot(df_model, p, city_order).loc[dates]
coords_arr = [CITY_COORDS[c] for c in city_order]
dist_mat = np.zeros((len(city_order), len(city_order)), dtype=np.float32)
for i in range(len(city_order)):
    lon1, lat1 = coords_arr[i]
    for j in range(len(city_order)):
        lon2, lat2 = coords_arr[j]
        if i == j:
            dist_mat[i, j] = 0.0
        else:
            dist_mat[i, j] = haversine(lon1, lat1, lon2, lat2)

def row_normalize_np(A, eps=1e-08):
    row_sum = A.sum(axis=1, keepdims=True)
    row_sum[row_sum < eps] = 1.0
    return (A / row_sum).astype(np.float32)

def build_static_graph(dist_mat, k_static=3, sigma=None, self_loop_weight=1.0, make_symmetric=True):
    non_zero_dist = dist_mat[dist_mat > 0]
    if sigma is None:
        if len(non_zero_dist) == 0:
            raise ValueError('距离矩阵中没有非零距离，无法构建静态图。')
        sigma = np.median(non_zero_dist)
    W_geo = np.exp(-dist_mat / (sigma + 1e-08)).astype(np.float32)
    np.fill_diagonal(W_geo, 0.0)
    A_raw = np.zeros_like(W_geo, dtype=np.float32)
    for i in range(W_geo.shape[0]):
        nbrs = np.argsort(W_geo[i])[::-1][:k_static]
        A_raw[i, nbrs] = W_geo[i, nbrs]
    if make_symmetric:
        A_raw = np.maximum(A_raw, A_raw.T).astype(np.float32)
    A_rw = A_raw.copy()
    np.fill_diagonal(A_rw, np.diag(A_rw) + self_loop_weight)
    A_rw = row_normalize_np(A_rw)
    return (A_raw.astype(np.float32), A_rw.astype(np.float32), float(sigma))
A_static_raw, A_static_rw, sigma_dist = build_static_graph(dist_mat=dist_mat, k_static=STATIC_K, sigma=None, self_loop_weight=STATIC_SELF_LOOP_WEIGHT, make_symmetric=STATIC_GRAPH_SYMMETRIC)
print('Geographical prior graph prepared; sample-wise dynamic adjacency is constructed from past AQI only.')
print('sigma_dist =', sigma_dist)
print('A_static_raw shape:', A_static_raw.shape)
print('A_static_rw shape:', A_static_rw.shape)
pd.DataFrame(dist_mat, index=city_order, columns=city_order).to_excel(os.path.join(SAVE_DIR, 'static_distance_matrix.xlsx'))
pd.DataFrame(A_static_raw, index=city_order, columns=city_order).to_excel(os.path.join(SAVE_DIR, 'static_adjacency_raw.xlsx'))
pd.DataFrame(A_static_rw, index=city_order, columns=city_order).to_excel(os.path.join(SAVE_DIR, 'static_adjacency_row_normalized.xlsx'))

def get_ceemdan():
    emd_base = EMD()
    ce = CEEMDAN(ext_EMD=emd_base, parallel=False)
    ce.trials = MAX_CEEMDAN_TRIALS
    ce.noise_width = CEEMDAN_NOISE_WIDTH
    ce.noise_seed(BASE_SEED)
    return ce

def decompose_series_ceemdan(series_1d):
    series_1d = np.asarray(series_1d, dtype=np.float64).reshape(-1)
    if series_1d.size < 3 or not np.isfinite(series_1d).all():
        raise ValueError('CEEMDAN 输入必须至少包含3个有限数值')
    ce = get_ceemdan()
    imfs = ce(series_1d)
    if imfs.ndim == 1:
        imfs = imfs[np.newaxis, :]
    residual = series_1d - imfs.sum(axis=0)
    return (imfs.astype(np.float32), residual.astype(np.float32))
if not CAUSAL_CEEMDAN:
    raise RuntimeError('本修订版仅允许严格因果滚动 CEEMDAN，禁止关闭 CAUSAL_CEEMDAN。')
component_records = []
city_component_list = []
C_dim_fixed = MAX_IMF_KEEP + int(USE_RESIDUAL_COMPONENT)
component_tensor = np.zeros((T_all, len(city_order), C_dim_fixed), dtype=np.float32)
for ci, city in enumerate(city_order):
    series = pivot_aqi.loc[dates, city].to_numpy(dtype=np.float32)
    city_components = np.zeros((C_dim_fixed, T_all), dtype=np.float32)
    for t in range(T_all):
        hist = series[max(0, t - history_len + 1):t + 1]
        if len(hist) < 3 or np.allclose(hist, hist[0]):
            city_components[-1 if USE_RESIDUAL_COMPONENT else 0, t] = hist[-1]
            continue
        imfs, residual = decompose_series_ceemdan(hist)
        n_keep = min(MAX_IMF_KEEP, imfs.shape[0])
        city_components[:n_keep, t] = imfs[:n_keep, -1]
        if USE_RESIDUAL_COMPONENT:
            city_components[-1, t] = hist[-1] - city_components[:MAX_IMF_KEEP, t].sum()
    component_tensor[:, ci, :] = city_components.T
    city_component_list.append(city_components)
    component_records.append({'City': city, 'Total_Used_Components': C_dim_fixed, 'Method': 'rolling-origin causal CEEMDAN, trailing history_len days', 'History_Length': history_len, 'No_Future_Observations': True})
max_comp_total = C_dim_fixed
component_summary_df = pd.DataFrame(component_records)
component_summary_df.to_excel(os.path.join(SAVE_DIR, 'component_summary.xlsx'), index=False)

def build_panel_and_check(df_model_local, col_list, date_list, city_list):
    panels = []
    for col in col_list:
        p = build_pivot(df_model_local, col, city_list).loc[date_list]
        if p.isna().any().any():
            raise ValueError(f'选中特征 {col} 在完整日期集上存在缺失，无法安全建模。')
        panels.append(p.values.astype(np.float32))
    return np.stack(panels, axis=-1)
exo_feature_cols = selected_features.copy()
X_exo_panel = build_panel_and_check(df_model, exo_feature_cols, dates, city_order)
y_panel = pivot_aqi.loc[dates, city_order].values.astype(np.float32)
y_poll_panel = np.stack([pollutant_pivots[p].loc[dates, city_order].values.astype(np.float32) for p in POLLUTANT_COLS], axis=-1)
C_dim = component_tensor.shape[-1]
F_dim = X_exo_panel.shape[-1]
print('X_comp_panel shape:', component_tensor.shape)
print('X_exo_panel shape:', X_exo_panel.shape)
print('y_panel shape:', y_panel.shape)
print('y_poll_panel shape:', y_poll_panel.shape)
comp_train = component_tensor[:train_end].reshape(-1, C_dim)
comp_center, comp_scale = fit_robust_scaler(comp_train)
component_tensor_std = transform_with_scaler(component_tensor.reshape(-1, C_dim), comp_center, comp_scale).reshape(T_all, len(city_order), C_dim)
exo_train = X_exo_panel[:train_end].reshape(-1, F_dim)
exo_center, exo_scale = fit_robust_scaler(exo_train)
X_exo_panel_std = transform_with_scaler(X_exo_panel.reshape(-1, F_dim), exo_center, exo_scale).reshape(T_all, len(city_order), F_dim)
skip_feature_names = [c for c in ['AQI', 'AQI_lag1'] if c in exo_feature_cols]
skip_feature_idx = [exo_feature_cols.index(c) for c in skip_feature_names]
print('Skip features:', skip_feature_names)

def build_dynamic_graph_from_history(aqi_history, geo_prior=None, top_k=STATIC_K, self_loop_weight=STATIC_SELF_LOOP_WEIGHT):
    H = np.asarray(aqi_history, dtype=np.float32)
    n_city = H.shape[1]
    if H.shape[0] < 3:
        corr = np.zeros((n_city, n_city), dtype=np.float32)
    else:
        corr = np.corrcoef(H.T)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    W = np.maximum(corr, 0.0)
    np.fill_diagonal(W, 0.0)
    if geo_prior is not None:
        W = 0.7 * W + 0.3 * np.asarray(geo_prior, dtype=np.float32)
        np.fill_diagonal(W, 0.0)
    A = np.zeros_like(W, dtype=np.float32)
    for i in range(n_city):
        order = np.argsort(W[i])[::-1]
        order = [j for j in order if j != i and W[i, j] > 0][:top_k]
        if order:
            A[i, order] = W[i, order]
    A = np.maximum(A, A.T).astype(np.float32)
    np.fill_diagonal(A, self_loop_weight)
    return row_normalize_np(A)

def build_samples(comp_panel_std, exo_panel_std, y_panel_raw, y_poll_panel_raw, dates, history_len, pred_horizon):
    Xc_list, Xe_list, Y_list, Ypoll_list, A_seq_list, date_list = ([], [], [], [], [], [])
    T = len(dates)
    for t in range(history_len, T - pred_horizon + 1):
        target_idx = t + pred_horizon - 1
        Xc_list.append(comp_panel_std[t - history_len:t])
        Xe_list.append(exo_panel_std[t - history_len:t])
        Y_list.append(y_panel_raw[target_idx])
        Ypoll_list.append(y_poll_panel_raw[target_idx])
        A_hist = []
        for h_idx in range(t - history_len, t):
            left = max(0, h_idx - history_len + 1)
            hist = y_panel_raw[left:h_idx + 1]
            A_hist.append(build_dynamic_graph_from_history(hist, geo_prior=A_static_raw))
        A_seq_list.append(np.stack(A_hist).astype(np.float32))
        date_list.append(dates[target_idx])
    Xc_arr = np.stack(Xc_list).astype(np.float32)
    Xe_arr = np.stack(Xe_list).astype(np.float32)
    Y_arr = np.stack(Y_list).astype(np.float32)
    Ypoll_arr = np.stack(Ypoll_list).astype(np.float32)
    A_arr = np.stack(A_seq_list).astype(np.float32)
    return (Xc_arr, Xe_arr, Y_arr, Ypoll_arr, A_arr, date_list)
Xc_all, Xe_all, Y_all_raw, Y_poll_all, A_all, target_dates = build_samples(comp_panel_std=component_tensor_std, exo_panel_std=X_exo_panel_std, y_panel_raw=y_panel, y_poll_panel_raw=y_poll_panel, dates=dates, history_len=history_len, pred_horizon=pred_horizon)
target_train_end = train_end - history_len - pred_horizon + 1
target_val_end = val_end - history_len - pred_horizon + 1
if target_train_end <= 0 or target_val_end <= target_train_end:
    raise ValueError('滑动窗口切分后训练/验证样本为空，请检查参数。')
Xc_train = Xc_all[:target_train_end]
Xe_train = Xe_all[:target_train_end]
Y_train_raw = Y_all_raw[:target_train_end]
Y_poll_train = Y_poll_all[:target_train_end]
A_train = A_all[:target_train_end]
dates_train_final = target_dates[:target_train_end]
Xc_val = Xc_all[target_train_end:target_val_end]
Xe_val = Xe_all[target_train_end:target_val_end]
Y_val_raw = Y_all_raw[target_train_end:target_val_end]
Y_poll_val = Y_poll_all[target_train_end:target_val_end]
A_val = A_all[target_train_end:target_val_end]
dates_val_final = target_dates[target_train_end:target_val_end]
Xc_test = Xc_all[target_val_end:]
Xe_test = Xe_all[target_val_end:]
Y_test_raw = Y_all_raw[target_val_end:]
Y_poll_test = Y_poll_all[target_val_end:]
A_test = A_all[target_val_end:]
dates_test_final = target_dates[target_val_end:]
print('Train samples:', len(Xc_train))
print('Val samples:', len(Xc_val))
print('Test samples:', len(Xc_test))
y_center, y_scale = fit_robust_scaler(Y_train_raw)
Y_train_std = transform_with_scaler(Y_train_raw, y_center, y_scale)

class DualBranchAQIDataset(Dataset):

    def __init__(self, Xc, Xe, Y=None, A=None):
        self.Xc = torch.tensor(Xc, dtype=torch.float32)
        self.Xe = torch.tensor(Xe, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32) if Y is not None else None
        self.A = torch.tensor(A, dtype=torch.float32) if A is not None else None

    def __len__(self):
        return len(self.Xc)

    def __getitem__(self, idx):
        if self.Y is None:
            return (self.Xc[idx], self.Xe[idx], self.A[idx])
        return (self.Xc[idx], self.Xe[idx], self.Y[idx], self.A[idx])


In [ ]:
RUN_ABLATION = True
ABLATION_VARIANTS = ['Full', 'w/o CEEMDAN Branch', 'w/o Exogenous Branch', 'w/o Graph Convolution', 'w/o Gated Fusion', 'w/o Skip Connection', 'w/o Event Loss']

def graph_mm(A, X):
    if A.dim() == 2:
        return torch.einsum('ij,bjf->bif', A, X)
    elif A.dim() == 3:
        return torch.einsum('bij,bjf->bif', A, X)
    else:
        raise ValueError('A dim must be 2 or 3')

class ChebConv(nn.Module):

    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.W0 = nn.Linear(in_feats, out_feats)
        self.W1 = nn.Linear(in_feats, out_feats)
        self.W2 = nn.Linear(in_feats, out_feats)

    def forward(self, X, A):
        T0 = X
        T1 = graph_mm(A, X)
        T2 = 2.0 * graph_mm(A, T1) - X
        return self.W0(T0) + self.W1(T1) + self.W2(T2)

class GatedTemporalBlock(nn.Module):

    def __init__(self, channels, kernel_size=3, dropout=0.1):
        super().__init__()
        self.filter_conv = nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=kernel_size // 2)
        self.gate_conv = nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=kernel_size // 2)
        self.norm = nn.BatchNorm1d(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        filt = torch.tanh(self.filter_conv(x))
        gate = torch.sigmoid(self.gate_conv(x))
        out = filt * gate
        out = self.norm(out)
        out = self.dropout(out)
        return out + x

class BranchEncoder(nn.Module):

    def __init__(self, in_feats, hidden, hist, dropout=0.1, use_graph=True):
        super().__init__()
        self.hist = hist
        self.hidden = hidden
        self.use_graph = use_graph
        self.in_proj = nn.Linear(in_feats, hidden)
        if self.use_graph:
            self.gconv1 = ChebConv(hidden, hidden)
            self.gconv2 = ChebConv(hidden, hidden)
        else:
            self.ffn1 = nn.Linear(hidden, hidden)
            self.ffn2 = nn.Linear(hidden, hidden)
        self.temporal1 = GatedTemporalBlock(hidden, kernel_size=3, dropout=dropout)
        self.temporal2 = GatedTemporalBlock(hidden, kernel_size=3, dropout=dropout)
        self.attn_fc = nn.Linear(hidden, max(16, hidden // 2))
        self.attn_score = nn.Linear(max(16, hidden // 2), 1)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(hidden)

    def forward(self, X_seq, A_seq):
        B, T, N, _ = X_seq.shape
        hs = []
        for t in range(T):
            xt = X_seq[:, t, :, :]
            At = A_seq[:, t, :, :]
            h = self.in_proj(xt)
            if self.use_graph:
                h = F.gelu(self.gconv1(h, At))
                h = self.dropout(h)
                h = F.gelu(self.gconv2(h, At))
                h = self.dropout(h)
            else:
                h = F.gelu(self.ffn1(h))
                h = self.dropout(h)
                h = F.gelu(self.ffn2(h))
                h = self.dropout(h)
            hs.append(h)
        H = torch.stack(hs, dim=1)
        H = H.permute(0, 2, 3, 1).contiguous().view(B * N, self.hidden, T)
        H = self.temporal1(H)
        H = self.temporal2(H)
        H = H.view(B, N, self.hidden, T).permute(0, 3, 1, 2).contiguous()
        H = self.ln(H)
        attn_logits = self.attn_score(torch.tanh(self.attn_fc(H))).squeeze(-1)
        attn = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        H_attn = (H * attn).sum(dim=1)
        H_last = H[:, -1, :, :]
        out = 0.6 * H_last + 0.4 * H_attn
        return out

class DualBranchCEGNet(nn.Module):

    def __init__(self, n_nodes, comp_dim, exo_dim, hidden, hist, skip_idx_list=None, dropout=0.1, use_graph=True, use_gated_fusion=True, use_skip=True):
        super().__init__()
        self.n_nodes = n_nodes
        self.hist = hist
        self.hidden = hidden
        self.skip_idx_list = skip_idx_list if skip_idx_list is not None else []
        self.use_gated_fusion = use_gated_fusion
        self.use_skip = use_skip
        self.comp_branch = BranchEncoder(comp_dim, hidden, hist, dropout=dropout, use_graph=use_graph)
        self.exo_branch = BranchEncoder(exo_dim, hidden, hist, dropout=dropout, use_graph=use_graph)
        self.comp_gate = nn.Sequential(nn.Linear(hidden, hidden), nn.Sigmoid())
        self.exo_gate = nn.Sequential(nn.Linear(hidden, hidden), nn.Sigmoid())
        self.fuse_gate = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.Sigmoid())
        if self.use_skip and len(self.skip_idx_list) > 0:
            self.skip_proj = nn.Linear(len(self.skip_idx_list), hidden // 4)
            skip_dim = hidden // 4
        else:
            self.skip_proj = None
            skip_dim = 0
        self.main_head = nn.Sequential(nn.Linear(hidden + skip_dim, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.event_head = nn.Sequential(nn.Linear(hidden + skip_dim, hidden // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden // 2, 1))
        self.calib_head = nn.Sequential(nn.Linear(hidden + skip_dim, hidden // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden // 2, 1))

    def forward(self, Xc_seq, Xe_seq, A_seq):
        hc = self.comp_branch(Xc_seq, A_seq)
        he = self.exo_branch(Xe_seq, A_seq)
        hc = hc * self.comp_gate(hc)
        he = he * self.exo_gate(he)
        if self.use_gated_fusion:
            g = self.fuse_gate(torch.cat([hc, he], dim=-1))
            h = g * hc + (1.0 - g) * he
        else:
            h = 0.5 * hc + 0.5 * he
        if self.skip_proj is not None:
            skip_x = Xe_seq[:, -1, :, self.skip_idx_list]
            skip_h = self.skip_proj(skip_x)
            h_cat = torch.cat([h, skip_h], dim=-1)
        else:
            h_cat = h
        main_out = self.main_head(h_cat).squeeze(-1)
        event_logit = self.event_head(h_cat).squeeze(-1)
        calib_bias = self.calib_head(h_cat).squeeze(-1)
        pred = main_out + 0.15 * calib_bias
        return (pred, event_logit)

def build_model(hidden, dropout, ablation_variant='Full'):
    use_graph = True
    use_gated_fusion = True
    use_skip = True
    if ablation_variant == 'w/o Graph Convolution':
        use_graph = False
    if ablation_variant == 'w/o Gated Fusion':
        use_gated_fusion = False
    if ablation_variant == 'w/o Skip Connection':
        use_skip = False
    return DualBranchCEGNet(n_nodes=len(city_order), comp_dim=C_dim, exo_dim=F_dim, hidden=hidden, hist=history_len, skip_idx_list=skip_feature_idx, dropout=dropout, use_graph=use_graph, use_gated_fusion=use_gated_fusion, use_skip=use_skip).to(device)

class MultiObjectiveDualLoss(nn.Module):

    def __init__(self, peak_thr, event_focus_thr=100.0, w_mse=0.4, w_huber=0.25, w_smape=0.15, w_event=0.2, peak_weight=2.5, delta=1.0, use_event_loss=True):
        super().__init__()
        self.peak_thr = float(peak_thr)
        self.event_focus_thr = float(event_focus_thr)
        self.w_mse = float(w_mse)
        self.w_huber = float(w_huber)
        self.w_smape = float(w_smape)
        self.w_event = float(w_event)
        self.peak_weight = float(peak_weight)
        self.delta = float(delta)
        self.use_event_loss = use_event_loss

    def forward(self, pred, event_logit, target):
        peak_w = torch.ones_like(target)
        peak_w[target >= self.peak_thr] = self.peak_weight
        mse = (peak_w * (pred - target) ** 2).mean()
        err = pred - target
        abs_err = torch.abs(err)
        delta_tensor = torch.tensor(self.delta, device=target.device, dtype=target.dtype)
        quad = torch.minimum(abs_err, delta_tensor)
        lin = abs_err - quad
        huber = (peak_w * (0.5 * quad ** 2 + self.delta * lin)).mean()
        smape = torch.mean(2.0 * torch.abs(pred - target) / (torch.abs(pred) + torch.abs(target) + 0.001))
        if self.use_event_loss:
            event_target = (target >= self.event_focus_thr).float()
            event_loss = F.binary_cross_entropy_with_logits(event_logit, event_target)
        else:
            event_loss = torch.tensor(0.0, device=target.device, dtype=target.dtype)
        total = self.w_mse * mse + self.w_huber * huber + self.w_smape * smape + self.w_event * event_loss
        return total

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for Xcb, Xeb, Yb, Ab in loader:
        Xcb = Xcb.to(device)
        Xeb = Xeb.to(device)
        Yb = Yb.to(device)
        Ab = Ab.to(device)
        optimizer.zero_grad()
        pred, event_logit = model(Xcb, Xeb, Ab)
        loss = criterion(pred, event_logit, Yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * Xcb.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, trues = ([], [])
    with torch.no_grad():
        for Xcb, Xeb, Yb, Ab in loader:
            Xcb = Xcb.to(device)
            Xeb = Xeb.to(device)
            Yb = Yb.to(device)
            Ab = Ab.to(device)
            pred, event_logit = model(Xcb, Xeb, Ab)
            loss = criterion(pred, event_logit, Yb)
            total_loss += loss.item() * Xcb.size(0)
            preds.append(pred.cpu().numpy())
            trues.append(Yb.cpu().numpy())
    preds = np.vstack(preds)
    trues = np.vstack(trues)
    return (total_loss / len(loader.dataset), preds, trues)

def predict_only(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for Xcb, Xeb, Ab in loader:
            Xcb = Xcb.to(device)
            Xeb = Xeb.to(device)
            Ab = Ab.to(device)
            pred, _ = model(Xcb, Xeb, Ab)
            preds.append(pred.cpu().numpy())
    return np.vstack(preds)


In [ ]:
def search_best_params_and_epoch(Xc_train, Xe_train, Y_train_std, A_train, trials=26, splits=4, ablation_variant='Full'):
    if len(Xc_train) < 20:
        raise ValueError(f'训练样本过少，无法搜索。当前样本数：{len(Xc_train)}')
    effective_splits = min(splits, len(Xc_train) - 1)
    if effective_splits < 2:
        raise ValueError('TimeSeriesSplit 的 n_splits 至少为 2。')
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    def objective(trial):
        hidden = trial.suggest_categorical('hidden', [64, 128, 256])
        lr = trial.suggest_float('lr', 0.0002, 0.003, log=True)
        wd = trial.suggest_float('wd', 1e-06, 8e-05, log=True)
        dropout = trial.suggest_float('dropout', 0.05, 0.25)
        fold_losses = []
        fold_best_epochs = []
        for tr_idx, va_idx in tscv.split(Xc_train):
            Xc_tr, Xe_tr, Y_tr, A_tr = (Xc_train[tr_idx], Xe_train[tr_idx], Y_train_std[tr_idx], A_train[tr_idx])
            Xc_va, Xe_va, Y_va, A_va = (Xc_train[va_idx], Xe_train[va_idx], Y_train_std[va_idx], A_train[va_idx])
            ds_tr = DualBranchAQIDataset(Xc_tr, Xe_tr, Y_tr, A_tr)
            ds_va = DualBranchAQIDataset(Xc_va, Xe_va, Y_va, A_va)
            dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=False)
            dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False)
            peak_thr = float(np.quantile(Y_tr.reshape(-1), PEAK_Q))
            criterion = MultiObjectiveDualLoss(peak_thr=peak_thr, event_focus_thr=EVENT_FOCUS_THR, w_mse=LOSS_W_MSE, w_huber=LOSS_W_HUBER, w_smape=LOSS_W_SMAPE, w_event=LOSS_W_EVENT, peak_weight=PEAK_WEIGHT, delta=HUBER_DELTA, use_event_loss=ablation_variant != 'w/o Event Loss')
            model = build_model(hidden=hidden, dropout=dropout, ablation_variant=ablation_variant)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
            best_loss = np.inf
            best_epoch = 1
            best_state = None
            wait = 0
            for epoch in range(1, max_search_epochs + 1):
                train_one_epoch(model, dl_tr, optimizer, criterion)
                va_loss, _, _ = evaluate(model, dl_va, criterion)
                if va_loss < best_loss:
                    best_loss = va_loss
                    best_epoch = epoch
                    best_state = copy.deepcopy(model.state_dict())
                    wait = 0
                else:
                    wait += 1
                    if wait >= search_patience:
                        break
            if best_state is not None:
                model.load_state_dict(best_state)
            fold_losses.append(best_loss)
            fold_best_epochs.append(best_epoch)
        trial.set_user_attr('best_epoch', int(np.median(fold_best_epochs)))
        return float(np.mean(fold_losses))
    set_global_seed(SEARCH_SEED)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=trials, show_progress_bar=False)
    best_params = study.best_params
    best_epoch = study.best_trial.user_attrs['best_epoch']
    best_epoch = int(min(max(best_epoch, MIN_FINAL_EPOCH), final_max_epochs_cap))
    return (best_params, best_epoch)

def fit_final_model_full_train(Xc_train, Xe_train, Y_train_std, A_train, best_params, best_epoch, seed=42, ablation_variant='Full'):
    set_global_seed(seed)
    ds = DualBranchAQIDataset(Xc_train, Xe_train, Y_train_std, A_train)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    peak_thr = float(np.quantile(Y_train_std.reshape(-1), PEAK_Q))
    criterion = MultiObjectiveDualLoss(peak_thr=peak_thr, event_focus_thr=EVENT_FOCUS_THR, w_mse=LOSS_W_MSE, w_huber=LOSS_W_HUBER, w_smape=LOSS_W_SMAPE, w_event=LOSS_W_EVENT, peak_weight=PEAK_WEIGHT, delta=HUBER_DELTA, use_event_loss=ablation_variant != 'w/o Event Loss')
    model = build_model(hidden=best_params['hidden'], dropout=best_params['dropout'], ablation_variant=ablation_variant)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=best_params['wd'])
    train_loss_list = []
    val_loss_stub = []
    for _ in range(best_epoch):
        tr_loss = train_one_epoch(model, dl, optimizer, criterion)
        train_loss_list.append(tr_loss)
        val_loss_stub.append(tr_loss)
    return (model, train_loss_list, val_loss_stub)

def fit_predict_ensemble(model_name, Xc_train, Xe_train, Y_train_std, A_train, Xc_val, Xe_val, A_val, Xc_test, Xe_test, A_test, ablation_variant='Full'):
    print(f'\n================ 搜索参数：{model_name} ================\n')
    best_params, best_epoch = search_best_params_and_epoch(Xc_train=Xc_train, Xe_train=Xe_train, Y_train_std=Y_train_std, A_train=A_train, trials=OPTUNA_TRIALS, splits=OPTUNA_SPLITS, ablation_variant=ablation_variant)
    print(f'{model_name} Best params:', best_params)
    print(f'{model_name} Best epoch:', best_epoch)
    val_pred_ds = DualBranchAQIDataset(Xc_val, Xe_val, None, A_val)
    test_pred_ds = DualBranchAQIDataset(Xc_test, Xe_test, None, A_test)
    val_pred_loader = DataLoader(val_pred_ds, batch_size=batch_size, shuffle=False)
    test_pred_loader = DataLoader(test_pred_ds, batch_size=batch_size, shuffle=False)
    val_preds_all = []
    test_preds_all = []
    model_save_dir = os.path.join(SAVE_DIR, safe_filename(model_name))
    ensure_dir(model_save_dir)
    for seed in ENSEMBLE_SEEDS:
        print(f'{model_name} | seed={seed}')
        model, train_loss_list, val_loss_list = fit_final_model_full_train(Xc_train=Xc_train, Xe_train=Xe_train, Y_train_std=Y_train_std, A_train=A_train, best_params=best_params, best_epoch=best_epoch, seed=seed, ablation_variant=ablation_variant)
        val_pred_std = predict_only(model, val_pred_loader)
        test_pred_std = predict_only(model, test_pred_loader)
        val_preds_all.append(val_pred_std)
        test_preds_all.append(test_pred_std)
        torch.save(model.state_dict(), os.path.join(model_save_dir, f'{safe_filename(model_name)}_seed_{seed}.pth'))
        save_curve(train_loss_list=train_loss_list, val_loss_list=val_loss_list, title=f'{model_name} Training Curve (seed={seed})', save_path=os.path.join(model_save_dir, f'{safe_filename(model_name)}_seed_{seed}_curve.png'))
    val_pred_std_mean = np.mean(np.stack(val_preds_all, axis=0), axis=0)
    test_pred_std_mean = np.mean(np.stack(test_preds_all, axis=0), axis=0)
    pd.DataFrame([best_params]).to_excel(os.path.join(model_save_dir, 'best_params.xlsx'), index=False)
    pd.DataFrame([{'best_epoch': best_epoch}]).to_excel(os.path.join(model_save_dir, 'best_epoch.xlsx'), index=False)
    return {'best_params': best_params, 'best_epoch': best_epoch, 'val_pred_std': val_pred_std_mean, 'test_pred_std': test_pred_std_mean}


In [ ]:
def city_bias_correction(y_val_true, pred_val, pred_test, method='median'):
    if method == 'median':
        bias = np.median(y_val_true - pred_val, axis=0)
    else:
        bias = np.mean(y_val_true - pred_val, axis=0)
    pred_val_bc = np.clip(pred_val + bias.reshape(1, -1), AQI_MIN, AQI_MAX).astype(np.float32)
    pred_test_bc = np.clip(pred_test + bias.reshape(1, -1), AQI_MIN, AQI_MAX).astype(np.float32)
    bias_df = pd.DataFrame({'City': city_order, 'Bias': bias})
    return (pred_val_bc, pred_test_bc, bias_df)

def prepare_ablation_inputs(variant_name):
    Xc_train_v, Xe_train_v, A_train_v = (Xc_train.copy(), Xe_train.copy(), A_train.copy())
    Xc_val_v, Xe_val_v, A_val_v = (Xc_val.copy(), Xe_val.copy(), A_val.copy())
    Xc_test_v, Xe_test_v, A_test_v = (Xc_test.copy(), Xe_test.copy(), A_test.copy())
    if variant_name == 'w/o CEEMDAN Branch':
        Xc_train_v = np.zeros_like(Xc_train_v, dtype=np.float32)
        Xc_val_v = np.zeros_like(Xc_val_v, dtype=np.float32)
        Xc_test_v = np.zeros_like(Xc_test_v, dtype=np.float32)
    elif variant_name == 'w/o Exogenous Branch':
        Xe_train_v = np.zeros_like(Xe_train_v, dtype=np.float32)
        Xe_val_v = np.zeros_like(Xe_val_v, dtype=np.float32)
        Xe_test_v = np.zeros_like(Xe_test_v, dtype=np.float32)
    return (Xc_train_v, Xe_train_v, A_train_v, Xc_val_v, Xe_val_v, A_val_v, Xc_test_v, Xe_test_v, A_test_v)
ablation_results = {}
ablation_pred_val = {}
ablation_pred_test = {}
ablation_param_rows = []
variants_to_run = ABLATION_VARIANTS if RUN_ABLATION else ['Full']
for variant in variants_to_run:
    print(f'\n\n================ 开始运行：{variant} ================\n')
    Xc_train_v, Xe_train_v, A_train_v, Xc_val_v, Xe_val_v, A_val_v, Xc_test_v, Xe_test_v, A_test_v = prepare_ablation_inputs(variant)
    result_v = fit_predict_ensemble(model_name=f'Ablation-{variant}', Xc_train=Xc_train_v, Xe_train=Xe_train_v, Y_train_std=Y_train_std, A_train=A_train_v, Xc_val=Xc_val_v, Xe_val=Xe_val_v, A_val=A_val_v, Xc_test=Xc_test_v, Xe_test=Xe_test_v, A_test=A_test_v, ablation_variant=variant)
    pred_val_v = np.clip(inverse_transform(result_v['val_pred_std'], y_center, y_scale), AQI_MIN, AQI_MAX).astype(np.float32)
    pred_test_v = np.clip(inverse_transform(result_v['test_pred_std'], y_center, y_scale), AQI_MIN, AQI_MAX).astype(np.float32)
    pred_val_v, pred_test_v, bias_df_v = city_bias_correction(Y_val_raw, pred_val_v, pred_test_v, method='median')
    bias_df_v.to_excel(os.path.join(SAVE_DIR, f'ablation_bias_{safe_filename(variant)}.xlsx'), index=False)
    ablation_results[variant] = result_v
    ablation_pred_val[variant] = pred_val_v
    ablation_pred_test[variant] = pred_test_v
    ablation_param_rows.append({'Variant': variant, **result_v['best_params'], 'best_epoch': result_v['best_epoch']})
ablation_best_params_df = pd.DataFrame(ablation_param_rows)
ablation_best_params_df.to_excel(os.path.join(SAVE_DIR, 'ablation_best_params.xlsx'), index=False)
main_result = ablation_results['Full']
pred_val = ablation_pred_val['Full']
pred_test = ablation_pred_test['Full']
main_bias_df = pd.read_excel(os.path.join(SAVE_DIR, f"ablation_bias_{safe_filename('Full')}.xlsx"))
main_bias_df.to_excel(os.path.join(SAVE_DIR, 'main_model_city_bias.xlsx'), index=False)


In [ ]:
def build_persistence_pred(y_panel, history_len, pred_horizon, start_idx, end_idx):
    preds = []
    for idx in range(start_idx, end_idx):
        t = history_len + idx
        target_idx = t + pred_horizon - 1
        prev_idx = target_idx - 1
        preds.append(y_panel[prev_idx])
    return np.stack(preds).astype(np.float32)

def build_ma_pred(y_panel, history_len, pred_horizon, start_idx, end_idx, window=7):
    preds = []
    for idx in range(start_idx, end_idx):
        t = history_len + idx
        target_idx = t + pred_horizon - 1
        left = max(0, target_idx - window)
        preds.append(y_panel[left:target_idx].mean(axis=0))
    return np.stack(preds).astype(np.float32)
val_start = target_train_end
val_end_idx = target_val_end
test_start = target_val_end
test_end_idx = len(target_dates)
pers_val = build_persistence_pred(y_panel, history_len, pred_horizon, val_start, val_end_idx)
pers_test = build_persistence_pred(y_panel, history_len, pred_horizon, test_start, test_end_idx)
ma7_val = build_ma_pred(y_panel, history_len, pred_horizon, val_start, val_end_idx, window=7)
ma7_test = build_ma_pred(y_panel, history_len, pred_horizon, test_start, test_end_idx, window=7)

def build_spatial_weights_from_static_graph(A_raw):
    A = np.asarray(A_raw, dtype=np.float64)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError('A_static_raw 必须是方阵')
    if np.any(A < 0) or not np.isfinite(A).all():
        raise ValueError('邻接矩阵含非法权重')
    strength = A.sum(axis=1)
    if strength.sum() <= 0:
        raise ValueError('图节点强度全为零')
    weights = strength / strength.sum()
    assert np.isclose(weights.sum(), 1.0)
    return (weights.astype(np.float32), strength.astype(np.float32))
w_space, node_strength = build_spatial_weights_from_static_graph(A_static_raw)
weight_df = pd.DataFrame({'City': city_order, 'Geographic_Node_Strength': node_strength, 'Weight_Final': w_space}).sort_values('Weight_Final', ascending=False)
weight_df.to_excel(os.path.join(SAVE_DIR, 'province_spatial_weights.xlsx'), index=False)
print('Province weights:', dict(zip(city_order, np.round(w_space, 5))))
print('Sum of weights:', float(w_space.sum()))

def province_series(y_city, weights=w_space):
    arr = np.asarray(y_city, dtype=np.float64)
    if arr.ndim != 2 or arr.shape[1] != len(weights):
        raise ValueError('省域汇总要求 [日期数, 城市数] 且城市顺序为city_order')
    return arr @ np.asarray(weights, dtype=np.float64)

def fit_citywise_fusion(y_val, val_pred_dict, test_pred_dict, city_order, alpha_candidates):
    best_alpha = None
    best_rmse = np.inf
    best_fused_val = None
    best_fused_test = None
    best_coef_df = None
    model_names = list(val_pred_dict.keys())
    for alpha in alpha_candidates:
        fused_val = np.zeros_like(y_val, dtype=np.float32)
        fused_test = np.zeros_like(next(iter(test_pred_dict.values())), dtype=np.float32)
        coef_rows = []
        for i, city in enumerate(city_order):
            Xv = np.column_stack([val_pred_dict[name][:, i] for name in model_names])
            Xt = np.column_stack([test_pred_dict[name][:, i] for name in model_names])
            reg = Ridge(alpha=alpha, fit_intercept=True)
            reg.fit(Xv, y_val[:, i])
            fused_val[:, i] = reg.predict(Xv)
            fused_test[:, i] = reg.predict(Xt)
            row = {'City': city, 'Intercept': float(reg.intercept_)}
            for j, name in enumerate(model_names):
                row[name] = float(reg.coef_[j])
            coef_rows.append(row)
        fused_val = np.clip(fused_val, AQI_MIN, AQI_MAX).astype(np.float32)
        fused_test = np.clip(fused_test, AQI_MIN, AQI_MAX).astype(np.float32)
        rmse_val = calc_metrics(y_val.reshape(-1), fused_val.reshape(-1))['RMSE']
        if rmse_val < best_rmse:
            best_rmse = rmse_val
            best_alpha = alpha
            best_fused_val = fused_val
            best_fused_test = fused_test
            best_coef_df = pd.DataFrame(coef_rows)
    return (best_alpha, best_fused_val, best_fused_test, best_coef_df)
candidate_val_pred = {'dual_branch_cegnet_style': pred_val, 'persistence': pers_val, 'ma7': ma7_val}
candidate_test_pred = {'dual_branch_cegnet_style': pred_test, 'persistence': pers_test, 'ma7': ma7_test}
best_fusion_alpha, fused_val, fused_test, fusion_coef_df = fit_citywise_fusion(y_val=Y_val_raw, val_pred_dict=candidate_val_pred, test_pred_dict=candidate_test_pred, city_order=city_order, alpha_candidates=FUSION_ALPHAS)
fusion_coef_df.to_excel(os.path.join(SAVE_DIR, 'fusion_coefficients.xlsx'), index=False)
print('Best fusion alpha:', best_fusion_alpha)
fused_val, fused_test, fusion_bias_df = city_bias_correction(Y_val_raw, fused_val, fused_test, method='median')
fusion_bias_df.to_excel(os.path.join(SAVE_DIR, 'fusion_city_bias.xlsx'), index=False)


In [ ]:
def overall_pollutant_consistency(y_true_aqi, y_true_poll, y_pred_aqi, phase_name, model_name):
    aqi_rebuild_true, _, _ = china_aqi_from_pollutants_np(y_true_poll)
    pred_vs_true = calc_metrics(y_true_aqi.reshape(-1), y_pred_aqi.reshape(-1))
    pred_vs_rebuild = calc_metrics(aqi_rebuild_true.reshape(-1), y_pred_aqi.reshape(-1))
    data_internal = calc_metrics(y_true_aqi.reshape(-1), aqi_rebuild_true.reshape(-1))
    grade_acc = float((aqi_grade_np(y_true_aqi).reshape(-1) == aqi_grade_np(y_pred_aqi).reshape(-1)).mean())
    return {'Model': model_name, 'Phase': phase_name, 'Pred_vs_TrueAQI_RMSE': pred_vs_true['RMSE'], 'Pred_vs_TrueAQI_MAE': pred_vs_true['MAE'], 'Pred_vs_RebuildTrueAQI_RMSE': pred_vs_rebuild['RMSE'], 'Pred_vs_RebuildTrueAQI_MAE': pred_vs_rebuild['MAE'], 'TrueAQI_vs_RebuildTrueAQI_RMSE': data_internal['RMSE'], 'TrueAQI_vs_RebuildTrueAQI_MAE': data_internal['MAE'], 'AQI_Grade_Acc': grade_acc}

def build_primary_pollutant_tables(y_true_aqi, y_true_poll, y_pred_aqi, phase_name, model_name):
    aqi_rebuild_true, true_primary_idx, _ = china_aqi_from_pollutants_np(y_true_poll)
    rows = []
    for j, p in enumerate(POLLUTANT_COLS):
        mask = true_primary_idx.reshape(-1) == j
        cnt = int(mask.sum())
        if cnt == 0:
            continue
        yt = y_true_aqi.reshape(-1)[mask]
        yp = y_pred_aqi.reshape(-1)[mask]
        yr = aqi_rebuild_true.reshape(-1)[mask]
        met_direct = calc_metrics(yt, yp)
        met_rebuild = calc_metrics(yr, yp)
        grade_acc = float((aqi_grade_np(yt) == aqi_grade_np(yp)).mean())
        rows.append({'Model': model_name, 'Phase': phase_name, 'PrimaryPollutant': p, 'Count': cnt, 'RMSE_TrueAQI': met_direct['RMSE'], 'MAE_TrueAQI': met_direct['MAE'], 'R2_TrueAQI': met_direct['R2'], 'RMSE_RebuildTrueAQI': met_rebuild['RMSE'], 'MAE_RebuildTrueAQI': met_rebuild['MAE'], 'AQI_Grade_Acc': grade_acc})
    return pd.DataFrame(rows)

def city_level_metrics(y_true, y_pred, city_names):
    rows = []
    for i, city in enumerate(city_names):
        met = calc_extended_metrics(y_true[:, i], y_pred[:, i], thresholds=EVENT_THRESHOLDS)
        rows.append({'City': city, **met})
    return pd.DataFrame(rows)

def summarize_metrics_rows(y_true, y_pred, split_name, model_name):
    overall = calc_extended_metrics(y_true.reshape(-1), y_pred.reshape(-1), thresholds=EVENT_THRESHOLDS)
    rows = [{'Model': model_name, 'Phase': split_name, 'City': 'Overall', **overall}]
    city_df = city_level_metrics(y_true, y_pred, city_order)
    city_df.insert(0, 'Phase', split_name)
    city_df.insert(0, 'Model', model_name)
    rows.extend(city_df.to_dict('records'))
    y_true_prov = province_series(y_true)
    y_pred_prov = province_series(y_pred)
    prov_met = calc_extended_metrics(y_true_prov, y_pred_prov, thresholds=EVENT_THRESHOLDS)
    prov_row = {'Model': model_name, 'Phase': split_name, 'City': 'Province', **prov_met}
    return (rows, prov_row)


In [ ]:
model_dict_eval = {'DualBranch-CEGNetStyle': {'val': pred_val, 'test': pred_test}, 'Fusion': {'val': fused_val, 'test': fused_test}, 'Persistence': {'val': pers_val, 'test': pers_test}, 'MA7': {'val': ma7_val, 'test': ma7_test}}
overall_rows, province_metric_rows = ([], [])
consistency_rows = []
primary_pollutant_rows = []
for model_name, pred_pack in model_dict_eval.items():
    for phase, y_true_aqi, y_true_poll, y_pred in [('Val', Y_val_raw, Y_poll_val, pred_pack['val']), ('Test', Y_test_raw, Y_poll_test, pred_pack['test'])]:
        rows, prov_row = summarize_metrics_rows(y_true_aqi, y_pred, phase, model_name)
        overall_rows.extend(rows)
        province_metric_rows.append(prov_row)
        consistency_rows.append(overall_pollutant_consistency(y_true_aqi=y_true_aqi, y_true_poll=y_true_poll, y_pred_aqi=y_pred, phase_name=phase, model_name=model_name))
        primary_pollutant_rows.append(build_primary_pollutant_tables(y_true_aqi=y_true_aqi, y_true_poll=y_true_poll, y_pred_aqi=y_pred, phase_name=phase, model_name=model_name))
overall_metrics_full_df = pd.DataFrame(overall_rows)
overall_metrics_df = overall_metrics_full_df[overall_metrics_full_df['City'] == 'Overall'].reset_index(drop=True)
city_metrics_df = overall_metrics_full_df[~overall_metrics_full_df['City'].isin(['Overall', 'Province'])].reset_index(drop=True)
province_metrics_df = pd.DataFrame(province_metric_rows)
consistency_df = pd.DataFrame(consistency_rows)
primary_pollutant_df = pd.concat(primary_pollutant_rows, axis=0, ignore_index=True)
round_cols = ['MSE', 'RMSE', 'MAE', 'MAPE', 'R2', 'MBE', 'SMAPE', 'EHR_100', 'Precision_100', 'FAR_100', 'CSI_100', 'AUC_100', 'ECE_100', 'EHR_150', 'Precision_150', 'FAR_150', 'CSI_150', 'AUC_150', 'ECE_150', 'EHR_200', 'Precision_200', 'FAR_200', 'CSI_200', 'AUC_200', 'ECE_200']
for df_tmp in [overall_metrics_df, city_metrics_df, province_metrics_df]:
    metric_cols = [c for c in round_cols if c in df_tmp.columns]
    df_tmp[metric_cols] = df_tmp[metric_cols].round(4)
for c in ['Pred_vs_TrueAQI_RMSE', 'Pred_vs_TrueAQI_MAE', 'Pred_vs_RebuildTrueAQI_RMSE', 'Pred_vs_RebuildTrueAQI_MAE', 'TrueAQI_vs_RebuildTrueAQI_RMSE', 'TrueAQI_vs_RebuildTrueAQI_MAE', 'AQI_Grade_Acc']:
    if c in consistency_df.columns:
        consistency_df[c] = consistency_df[c].round(4)
for c in ['RMSE_TrueAQI', 'MAE_TrueAQI', 'R2_TrueAQI', 'RMSE_RebuildTrueAQI', 'MAE_RebuildTrueAQI', 'AQI_Grade_Acc']:
    if c in primary_pollutant_df.columns:
        primary_pollutant_df[c] = primary_pollutant_df[c].round(4)


In [ ]:
ablation_overall_rows = []
ablation_province_rows = []
ablation_city_rows = []
for variant in variants_to_run:
    for phase, y_true, y_pred in [('Val', Y_val_raw, ablation_pred_val[variant]), ('Test', Y_test_raw, ablation_pred_test[variant])]:
        overall_met = calc_extended_metrics(y_true.reshape(-1), y_pred.reshape(-1), thresholds=EVENT_THRESHOLDS)
        ablation_overall_rows.append({'Variant': variant, 'Phase': phase, **overall_met})
        prov_true = province_series(y_true)
        prov_pred = province_series(y_pred)
        prov_met = calc_extended_metrics(prov_true, prov_pred, thresholds=EVENT_THRESHOLDS)
        ablation_province_rows.append({'Variant': variant, 'Phase': phase, **prov_met})
        city_df_tmp = city_level_metrics(y_true, y_pred, city_order)
        city_df_tmp.insert(0, 'Phase', phase)
        city_df_tmp.insert(0, 'Variant', variant)
        ablation_city_rows.append(city_df_tmp)
ablation_overall_df = pd.DataFrame(ablation_overall_rows)
ablation_province_df = pd.DataFrame(ablation_province_rows)
ablation_city_df = pd.concat(ablation_city_rows, axis=0, ignore_index=True)
for df_tmp in [ablation_overall_df, ablation_province_df, ablation_city_df]:
    metric_cols = [c for c in round_cols if c in df_tmp.columns]
    df_tmp[metric_cols] = df_tmp[metric_cols].round(4)
ablation_test_rmse_rank = ablation_overall_df[ablation_overall_df['Phase'] == 'Test'].sort_values('RMSE').reset_index(drop=True)
ablation_test_r2_rank = ablation_overall_df[ablation_overall_df['Phase'] == 'Test'].sort_values('R2', ascending=False).reset_index(drop=True)
ablation_test_ehr100_rank = ablation_overall_df[ablation_overall_df['Phase'] == 'Test'].sort_values('EHR_100', ascending=False).reset_index(drop=True)


In [ ]:
METRIC_GROUPS = {'basic_metrics': ['MSE', 'RMSE', 'MAE', 'MAPE', 'R2', 'MBE', 'SMAPE'], 'event_100_metrics': ['EHR_100', 'Precision_100', 'FAR_100', 'CSI_100', 'AUC_100', 'ECE_100'], 'event_150_metrics': ['EHR_150', 'Precision_150', 'FAR_150', 'CSI_150', 'AUC_150', 'ECE_150'], 'event_200_metrics': ['EHR_200', 'Precision_200', 'FAR_200', 'CSI_200', 'AUC_200', 'ECE_200']}
CONSISTENCY_GROUPS = {'consistency_core': ['Pred_vs_TrueAQI_RMSE', 'Pred_vs_TrueAQI_MAE', 'Pred_vs_RebuildTrueAQI_RMSE', 'Pred_vs_RebuildTrueAQI_MAE', 'TrueAQI_vs_RebuildTrueAQI_RMSE', 'TrueAQI_vs_RebuildTrueAQI_MAE', 'AQI_Grade_Acc']}
PRIMARY_POLLUTANT_GROUPS = {'primary_pollutant_metrics': ['Count', 'RMSE_TrueAQI', 'MAE_TrueAQI', 'R2_TrueAQI', 'RMSE_RebuildTrueAQI', 'MAE_RebuildTrueAQI', 'AQI_Grade_Acc']}

def print_grouped_tables(df, id_cols, groups, title, sort_cols=None, ascending=True):
    print(f"\n{'=' * 24} {title} {'=' * 24}\n")
    df_show = df.copy()
    if sort_cols is not None:
        df_show = df_show.sort_values(sort_cols, ascending=ascending)
    for group_name, metric_cols in groups.items():
        use_cols = id_cols + [c for c in metric_cols if c in df_show.columns]
        sub_df = df_show[use_cols].copy()
        print(f'\n----- {group_name} -----\n')
        print(sub_df.to_string(index=False))

def save_grouped_sheets(df, id_cols, groups, writer, prefix, sort_cols=None, ascending=True):
    df_save = df.copy()
    if sort_cols is not None:
        df_save = df_save.sort_values(sort_cols, ascending=ascending)
    for group_name, metric_cols in groups.items():
        use_cols = id_cols + [c for c in metric_cols if c in df_save.columns]
        sub_df = df_save[use_cols].copy()
        sheet_name = f'{prefix}_{group_name}'[:31]
        sub_df.to_excel(writer, sheet_name=sheet_name, index=False)
print_grouped_tables(df=overall_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, title='Overall Metrics', sort_cols=['Phase', 'RMSE'])
print_grouped_tables(df=province_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, title='Province Metrics', sort_cols=['Phase', 'RMSE'])
print_grouped_tables(df=consistency_df, id_cols=['Model', 'Phase'], groups=CONSISTENCY_GROUPS, title='Pollutant Consistency Metrics', sort_cols=['Phase', 'Pred_vs_RebuildTrueAQI_RMSE'])
print_grouped_tables(df=primary_pollutant_df, id_cols=['Model', 'Phase', 'PrimaryPollutant'], groups=PRIMARY_POLLUTANT_GROUPS, title='Primary Pollutant Grouped Metrics', sort_cols=['Phase', 'PrimaryPollutant', 'RMSE_TrueAQI'])
print_grouped_tables(df=ablation_overall_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, title='Ablation Overall Metrics', sort_cols=['Phase', 'RMSE'])
print_grouped_tables(df=ablation_province_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, title='Ablation Province Metrics', sort_cols=['Phase', 'RMSE'])
print('\n================ Test RMSE Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('RMSE').to_string(index=False))
print('\n================ Test RMSE Ranking (Province) ================\n')
print(province_metrics_df[province_metrics_df['Phase'] == 'Test'].sort_values('RMSE').to_string(index=False))
print('\n================ Test EHR@100 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('EHR_100', ascending=False).to_string(index=False))
print('\n================ Test EHR@150 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('EHR_150', ascending=False).to_string(index=False))
print('\n================ Test EHR@200 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('EHR_200', ascending=False).to_string(index=False))
print('\n================ Test AUC@100 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('AUC_100', ascending=False).to_string(index=False))
print('\n================ Test AUC@150 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('AUC_150', ascending=False).to_string(index=False))
print('\n================ Test AUC@200 Ranking (Overall) ================\n')
print(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('AUC_200', ascending=False).to_string(index=False))
print('\n================ Test Consistency Ranking ================\n')
print(consistency_df[consistency_df['Phase'] == 'Test'].sort_values('Pred_vs_RebuildTrueAQI_RMSE').to_string(index=False))
print('\n================ Ablation Test RMSE Ranking ================\n')
print(ablation_test_rmse_rank.to_string(index=False))
print('\n================ Ablation Test R2 Ranking ================\n')
print(ablation_test_r2_rank.to_string(index=False))
print('\n================ Ablation Test EHR@100 Ranking ================\n')
print(ablation_test_ehr100_rank.to_string(index=False))


In [ ]:
city_records = []
prov_records = []
for phase, y_true, y_true_poll, dates_phase in [('验证集', Y_val_raw, Y_poll_val, dates_val_final), ('测试集', Y_test_raw, Y_poll_test, dates_test_final)]:
    aqi_rebuild_true, primary_idx_true, _ = china_aqi_from_pollutants_np(y_true_poll)
    for model_name, pred_pack in model_dict_eval.items():
        y_pred = pred_pack['val'] if phase == '验证集' else pred_pack['test']
        for i, city in enumerate(city_order):
            for k, dt in enumerate(dates_phase):
                city_records.append({'Phase': phase, 'Model': model_name, 'Date': dt, 'City': city, 'True_AQI': float(y_true[k, i]), 'Pred_AQI': float(y_pred[k, i]), 'True_AQI_Rebuild_FromPollutants': float(aqi_rebuild_true[k, i]), 'True_PrimaryPollutant': POLLUTANT_COLS[int(primary_idx_true[k, i])]})
        y_true_prov = province_series(y_true)
        y_pred_prov = province_series(y_pred)
        for dt, yt, yp in zip(dates_phase, y_true_prov, y_pred_prov):
            prov_records.append({'Phase': phase, 'Model': model_name, 'Date': dt, 'True_Prov_AQI': float(yt), 'Pred_Prov_AQI': float(yp)})
city_pred_df = pd.DataFrame(city_records)
prov_pred_df = pd.DataFrame(prov_records)
result_excel_path = os.path.join(SAVE_DIR, 'full_results_tables_split.xlsx')
with pd.ExcelWriter(result_excel_path, engine='openpyxl') as writer:
    save_grouped_sheets(df=overall_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='overall', sort_cols=['Phase', 'RMSE'])
    save_grouped_sheets(df=province_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='province', sort_cols=['Phase', 'RMSE'])
    save_grouped_sheets(df=city_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='city', sort_cols=['Phase', 'City'])
    save_grouped_sheets(df=consistency_df, id_cols=['Model', 'Phase'], groups=CONSISTENCY_GROUPS, writer=writer, prefix='consistency', sort_cols=['Phase', 'Pred_vs_RebuildTrueAQI_RMSE'])
    save_grouped_sheets(df=primary_pollutant_df, id_cols=['Model', 'Phase', 'PrimaryPollutant'], groups=PRIMARY_POLLUTANT_GROUPS, writer=writer, prefix='primary_pollutant', sort_cols=['Phase', 'PrimaryPollutant', 'RMSE_TrueAQI'])
    save_grouped_sheets(df=ablation_overall_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, writer=writer, prefix='ablation_overall', sort_cols=['Phase', 'RMSE'])
    save_grouped_sheets(df=ablation_province_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, writer=writer, prefix='ablation_province', sort_cols=['Phase', 'RMSE'])
    save_grouped_sheets(df=ablation_city_df, id_cols=['Variant', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='ablation_city', sort_cols=['Phase', 'City'])
    overall_metrics_df.to_excel(writer, sheet_name='overall_all', index=False)
    province_metrics_df.to_excel(writer, sheet_name='province_all', index=False)
    consistency_df.to_excel(writer, sheet_name='consistency_all', index=False)
    primary_pollutant_df.to_excel(writer, sheet_name='primary_pollutant_all', index=False)
    ablation_overall_df.to_excel(writer, sheet_name='ablation_overall_all', index=False)
    ablation_province_df.to_excel(writer, sheet_name='ablation_province_all', index=False)
    ablation_best_params_df.to_excel(writer, sheet_name='ablation_best_params', index=False)
    overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('RMSE').to_excel(writer, sheet_name='test_rmse_rank_overall', index=False)
    province_metrics_df[province_metrics_df['Phase'] == 'Test'].sort_values('RMSE').to_excel(writer, sheet_name='test_rmse_rank_province', index=False)
    consistency_df[consistency_df['Phase'] == 'Test'].sort_values('Pred_vs_RebuildTrueAQI_RMSE').to_excel(writer, sheet_name='test_consistency_rank', index=False)
    ablation_test_rmse_rank.to_excel(writer, sheet_name='ablation_test_rmse_rank', index=False)
    ablation_test_r2_rank.to_excel(writer, sheet_name='ablation_test_r2_rank', index=False)
    ablation_test_ehr100_rank.to_excel(writer, sheet_name='ablation_test_ehr100_rank', index=False)
    city_pred_df.to_excel(writer, sheet_name='city_predictions', index=False)
    prov_pred_df.to_excel(writer, sheet_name='province_predictions', index=False)
    fusion_coef_df.to_excel(writer, sheet_name='fusion_coefficients', index=False)
    fusion_bias_df.to_excel(writer, sheet_name='fusion_city_bias', index=False)
    weight_df.to_excel(writer, sheet_name='province_weights', index=False)
    main_bias_df.to_excel(writer, sheet_name='main_model_city_bias', index=False)
    mrmr_rank_df.to_excel(writer, sheet_name='mrmr_ranking', index=False)
    stability_df.to_excel(writer, sheet_name='stability_selection', index=False)
    surrogate_metrics_df.to_excel(writer, sheet_name='fs_surrogate_metrics', index=False)
    pd.DataFrame({'Selected_Features': selected_features}).to_excel(writer, sheet_name='selected_features', index=False)
    selection_source_df.to_excel(writer, sheet_name='selected_feature_sources', index=False)
    component_summary_df.to_excel(writer, sheet_name='component_summary', index=False)
    pd.DataFrame(dist_mat, index=city_order, columns=city_order).to_excel(writer, sheet_name='static_dist_matrix')
    pd.DataFrame(A_static_raw, index=city_order, columns=city_order).to_excel(writer, sheet_name='static_adj_raw')
    pd.DataFrame(A_static_rw, index=city_order, columns=city_order).to_excel(writer, sheet_name='static_adj_rw')
plot_model_order = ['DualBranch-CEGNetStyle', 'Fusion', 'Persistence']
for phase_name, y_true, dates_phase, key in [('Val', Y_val_raw, dates_val_final, 'val'), ('Test', Y_test_raw, dates_test_final, 'test')]:
    plt.figure(figsize=(14, 6))
    plt.plot(dates_phase, province_series(y_true), label='True Province AQI', linewidth=2.5)
    for model_name in plot_model_order:
        plt.plot(dates_phase, province_series(model_dict_eval[model_name][key]), label=model_name)
    plt.title(f'Province Level Comparison ({phase_name})')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'province_comparison_{phase_name.lower()}.png'), dpi=220)
    plt.show()
plt.figure(figsize=(10, 5))
plt.bar(overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('RMSE')['Model'], overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].sort_values('RMSE')['RMSE'])
plt.xticks(rotation=20)
plt.ylabel('RMSE')
plt.title('Overall Test RMSE Ranking')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overall_test_rmse_bar.png'), dpi=220)
plt.show()
pp_plot = primary_pollutant_df[primary_pollutant_df['Phase'] == 'Test'].copy()
if not pp_plot.empty:
    for pollutant in pp_plot['PrimaryPollutant'].unique():
        sub = pp_plot[pp_plot['PrimaryPollutant'] == pollutant].copy()
        plt.figure(figsize=(8, 4))
        plt.bar(sub['Model'], sub['RMSE_TrueAQI'])
        plt.xticks(rotation=20)
        plt.ylabel('RMSE')
        plt.title(f'Test AQI RMSE | True Primary Pollutant = {pollutant}')
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, f'primary_pollutant_{safe_filename(pollutant)}_test_rmse_bar.png'), dpi=220)
        plt.show()
plt.figure(figsize=(12, 6))
plt.bar(ablation_test_rmse_rank['Variant'], ablation_test_rmse_rank['RMSE'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('RMSE')
plt.title('Ablation Study - Test RMSE Ranking')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ablation_test_rmse_ranking.png'), dpi=220)
plt.show()
plt.figure(figsize=(12, 6))
plt.bar(ablation_test_ehr100_rank['Variant'], ablation_test_ehr100_rank['EHR_100'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('EHR@100')
plt.title('Ablation Study - Test EHR@100 Ranking')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ablation_test_ehr100_ranking.png'), dpi=220)
plt.show()
summary_note = {'model_name': 'DualBranch CEG-Net Style', 'prediction_protocol': 'Predictions are the arithmetic mean across ENSEMBLE_SEEDS; do not describe these outputs as a single-seed single-model prediction.', 'ensemble_seeds': ENSEMBLE_SEEDS, 'branch_1': 'AQI CEEMDAN decomposition branch', 'branch_2': 'Selected multivariate exogenous feature branch', 'graph_type': 'static geographical prior + sample-wise causal historical-correlation graph', 'primary_target': 'geographically weighted province-wide urban AQI composite (not official AQI)', 'province_weight_method': 'raw symmetric geographic adjacency node strength', 'ceemdan_method': 'causal rolling trailing-window CEEMDAN', 'use_winsorize': USE_WINSORIZE, 'feature_selector': 'time_safe_mrmr_plus_block_stability_selection', 'event_thresholds': EVENT_THRESHOLDS, 'run_ablation': RUN_ABLATION, 'ablation_variants': ABLATION_VARIANTS, 'main_points': ['AQI 分解分支与外生变量分支独立编码后门控融合', '使用 Cheb 图卷积 + 门控时序卷积，贴近 CEG-Net 风格', '默认关闭极端值裁剪，保留事件峰值信息', '保留多变量特征筛选，避免把噪声变量全部送入模型', '训练损失同时优化回归误差与事件识别能力', '输出 AQI 100/150/200 三个阈值下的事件指标', '主模型结果、消融实验结果、关键污染物一致性分析均按分组表展示与保存']}
with open(os.path.join(SAVE_DIR, 'run_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary_note, f, ensure_ascii=False, indent=2)
print('\n全部运行完成，结果已保存到：', SAVE_DIR)
print('分组汇总文件：', result_excel_path)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
TIME_TICK_INTERVAL = 3
SET_NAME_MAP = {'Validation': 'Set1', 'Test': 'Set2'}
PALETTE = {'observed': '#E79BB2', 'dual': '#7ECBEF', 'fusion': '#8FD8B8', 'persistence': '#F6C38B', 'grid': '#EAF7F1', 'spine': '#D7EEE5', 'text': '#4F6475', 'fill_dual': '#DFF4FD'}
plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 320, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'axes.unicode_minus': False, 'axes.labelsize': 14, 'axes.titlesize': 16, 'xtick.labelsize': 11.5, 'ytick.labelsize': 11.5, 'legend.fontsize': 11.5, 'axes.linewidth': 1.1, 'pdf.fonttype': 42, 'ps.fonttype': 42})

def save_fig(fig, save_dir, name):
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(os.path.join(save_dir, f'{name}.png'), bbox_inches='tight', dpi=320)
    fig.savefig(os.path.join(save_dir, f'{name}.pdf'), bbox_inches='tight')
    plt.show()

def beautify_axis(ax, dates_phase):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(PALETTE['spine'])
    ax.spines['bottom'].set_color(PALETTE['spine'])
    ax.grid(axis='y', linestyle='--', linewidth=0.9, color=PALETTE['grid'], alpha=0.9)
    ax.set_axisbelow(True)
    locator = mdates.MonthLocator(interval=TIME_TICK_INTERVAL)
    formatter = mdates.DateFormatter('%Y-%m')
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xlim(min(dates_phase), max(dates_phase))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', color=PALETTE['text'])
    ax.tick_params(axis='y', which='both', labelleft=False, left=False)
    ax.tick_params(axis='x', which='major', labelbottom=True, bottom=True, colors=PALETTE['text'], length=3.5, width=0.8)

def add_soft_line(ax, x, y, color, lw=2.2, ls='-', alpha=1.0, z=3, label=None):
    line, = ax.plot(x, y, color=color, linewidth=lw, linestyle=ls, alpha=alpha, zorder=z, label=label, solid_capstyle='round', solid_joinstyle='round')
    line.set_path_effects([pe.Stroke(linewidth=lw + 1.6, foreground='white', alpha=0.95), pe.Normal()])
    return line

def plot_province_comparison_fresh(phase_name, y_true, dates_phase, key, save_dir):
    y_obs = province_series(y_true)
    y_dual = province_series(model_dict_eval['DualBranch-CEGNetStyle'][key])
    y_fusion = province_series(model_dict_eval['Fusion'][key])
    y_persist = province_series(model_dict_eval['Persistence'][key])
    fig, ax = plt.subplots(figsize=(12.6, 5.2))
    add_soft_line(ax, dates_phase, y_persist, color=PALETTE['persistence'], lw=1.8, ls='--', alpha=0.95, z=2, label='Persistence')
    add_soft_line(ax, dates_phase, y_fusion, color=PALETTE['fusion'], lw=2.0, ls='-.', alpha=0.98, z=3, label='Fusion')
    add_soft_line(ax, dates_phase, y_obs, color=PALETTE['observed'], lw=2.25, ls='-', alpha=0.98, z=4, label='Observed')
    add_soft_line(ax, dates_phase, y_dual, color=PALETTE['dual'], lw=2.9, ls='-', alpha=1.0, z=5, label='DualBranch-CEGNetStyle')
    ax.fill_between(dates_phase, y_obs, y_dual, color=PALETTE['fill_dual'], alpha=0.1, zorder=1)
    beautify_axis(ax, dates_phase)
    ax.set_ylabel('AQI', color=PALETTE['text'])
    ax.set_xlabel('Date', color=PALETTE['text'])
    display_phase = SET_NAME_MAP.get(phase_name, phase_name)
    ax.set_title(f'Province-level AQI Prediction on {display_phase}', color=PALETTE['text'], pad=14)
    handles, labels = ax.get_legend_handles_labels()
    order_map = {'Observed': 0, 'DualBranch-CEGNetStyle': 1, 'Fusion': 2, 'Persistence': 3}
    order = np.argsort([order_map[l] for l in labels])
    handles = [handles[i] for i in order]
    labels = [labels[i] for i in order]
    leg = ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.17), ncol=4, frameon=False, handlelength=2.8, columnspacing=1.5)
    for txt in leg.get_texts():
        txt.set_color(PALETTE['text'])
    fig.tight_layout()
    save_fig(fig, save_dir, f'province_comparison_{phase_name.lower()}_fresh')

def plot_dualbranch_fit_fresh(phase_name, y_true, y_pred, dates_phase, save_dir):
    y_obs = province_series(y_true)
    y_hat = province_series(y_pred)
    fig, ax = plt.subplots(figsize=(12.4, 5.0))
    ax.fill_between(dates_phase, y_obs, y_hat, color=PALETTE['fill_dual'], alpha=0.18, zorder=1)
    add_soft_line(ax, dates_phase, y_obs, color=PALETTE['observed'], lw=2.35, ls='-', alpha=0.98, z=4, label='Observed')
    add_soft_line(ax, dates_phase, y_hat, color=PALETTE['dual'], lw=3.0, ls='-', alpha=1.0, z=5, label='Predicted (DualBranch-CEGNetStyle)')
    beautify_axis(ax, dates_phase)
    ax.set_ylabel('AQI', color=PALETTE['text'])
    ax.set_xlabel('Date', color=PALETTE['text'])
    display_phase = SET_NAME_MAP.get(phase_name, phase_name)
    ax.set_title(f'DualBranch-CEGNetStyle Fit Against Observations on {display_phase}', color=PALETTE['text'], pad=14)
    leg = ax.legend(loc='upper right', frameon=False, handlelength=2.8)
    for txt in leg.get_texts():
        txt.set_color(PALETTE['text'])
    fig.tight_layout()
    save_fig(fig, save_dir, f'dualbranch_province_fit_{phase_name.lower()}_fresh')

def plot_dualbranch_scatter_fresh(phase_name, y_true, y_pred, save_dir):
    y_obs = province_series(y_true).reshape(-1)
    y_hat = province_series(y_pred).reshape(-1)
    low = min(np.min(y_obs), np.min(y_hat))
    high = max(np.max(y_obs), np.max(y_hat))
    pad = (high - low) * 0.05
    low -= pad
    high += pad
    fig, ax = plt.subplots(figsize=(5.8, 5.4))
    ax.scatter(y_obs, y_hat, s=42, color=PALETTE['dual'], alpha=0.78, edgecolors='white', linewidths=0.7, zorder=3)
    ax.plot([low, high], [low, high], linestyle='--', linewidth=1.4, color=PALETTE['observed'], alpha=0.9, zorder=2)
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_aspect('equal', adjustable='box')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(PALETTE['spine'])
    ax.spines['bottom'].set_color(PALETTE['spine'])
    ax.grid(True, linestyle='--', linewidth=0.9, color=PALETTE['grid'], alpha=0.9)
    ax.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)
    ax.set_xlabel('Observed AQI', color=PALETTE['text'])
    ax.set_ylabel('Predicted AQI', color=PALETTE['text'])
    display_phase = SET_NAME_MAP.get(phase_name, phase_name)
    ax.set_title(f'DualBranch-CEGNetStyle Regression Fit on {display_phase}', color=PALETTE['text'], pad=12)
    fig.tight_layout()
    save_fig(fig, save_dir, f'dualbranch_scatter_fit_{phase_name.lower()}_fresh')
for phase_name, y_true, dates_phase, key in [('Validation', Y_val_raw, dates_val_final, 'val'), ('Test', Y_test_raw, dates_test_final, 'test')]:
    plot_province_comparison_fresh(phase_name=phase_name, y_true=y_true, dates_phase=dates_phase, key=key, save_dir=SAVE_DIR)
    plot_dualbranch_fit_fresh(phase_name=phase_name, y_true=y_true, y_pred=model_dict_eval['DualBranch-CEGNetStyle'][key], dates_phase=dates_phase, save_dir=SAVE_DIR)
    plot_dualbranch_scatter_fresh(phase_name=phase_name, y_true=y_true, y_pred=model_dict_eval['DualBranch-CEGNetStyle'][key], save_dir=SAVE_DIR)
print('新版小清新论文图已保存到：', SAVE_DIR)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates
CEEMDAN_PLOT_DIR = os.path.join(SAVE_DIR, 'ceemdan_pretty_plots')
os.makedirs(CEEMDAN_PLOT_DIR, exist_ok=True)
CITY_NAME_EN = {'长春': 'Changchun', '吉林': 'Jilin', '四平': 'Siping', '辽源': 'Liaoyuan', '通化': 'Tonghua', '白山': 'Baishan', '松原': 'Songyuan', '白城': 'Baicheng', '延边': 'Yanbian'}
CEEMDAN_COLORS = {'signal': '#E79BB2', 'residual': '#C7A6F5', 'grid': '#EAF7F1', 'spine': '#D7EEE5', 'text': '#4F6475'}
IMF_PASTEL_COLORS = ['#7ECBEF', '#8FD8B8', '#F6C38B', '#A9B8F8', '#9EDDD6', '#F4B5A6', '#B8E39A', '#DAB6F8']
plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 320, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'axes.unicode_minus': False, 'axes.labelsize': 13, 'axes.titlesize': 15, 'xtick.labelsize': 10.0, 'ytick.labelsize': 10.0, 'pdf.fonttype': 42, 'ps.fonttype': 42})

def safe_filename(name: str) -> str:
    invalid_chars = '\\\\/:*?"<>|'
    for ch in invalid_chars:
        name = name.replace(ch, '_')
    return str(name)

def _style_ceemdan_axis(ax, show_x=False):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(CEEMDAN_COLORS['spine'])
    ax.spines['bottom'].set_color(CEEMDAN_COLORS['spine'])
    ax.grid(axis='y', linestyle='--', linewidth=0.7, color=CEEMDAN_COLORS['grid'], alpha=0.85)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', which='both', labelleft=False, left=False)
    if show_x:
        locator = mdates.MonthLocator(interval=4)
        formatter = mdates.DateFormatter('%Y-%m')
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(formatter)
        ax.tick_params(axis='x', which='major', labelbottom=True, bottom=True, labelsize=10, colors=CEEMDAN_COLORS['text'], rotation=35)
        for label in ax.get_xticklabels():
            label.set_ha('right')
    else:
        ax.tick_params(axis='x', which='both', labelbottom=False, bottom=False)

def _draw_pretty_line(ax, x, y, color, label_text, lw=1.5, fill_alpha=0.1):
    line, = ax.plot(x, y, color=color, linewidth=lw, zorder=3, solid_capstyle='round', solid_joinstyle='round')
    line.set_path_effects([pe.Stroke(linewidth=lw + 0.8, foreground='white', alpha=0.92), pe.Normal()])
    ax.fill_between(x, 0, y, color=color, alpha=fill_alpha, zorder=1)
    ax.axhline(0, color='#F7FBF8', linewidth=0.8, zorder=0)
    ax.text(0.012, 0.82, label_text, transform=ax.transAxes, ha='left', va='center', fontsize=10.2, color=CEEMDAN_COLORS['text'], bbox=dict(boxstyle='round,pad=0.22', facecolor='white', edgecolor=color, linewidth=0.9, alpha=0.92))

def plot_city_ceemdan_from_current_script(city_name, city_idx):
    city_name_en = CITY_NAME_EN.get(city_name, city_name)
    signal = pivot_aqi.loc[dates, city_name].values.astype(np.float32)
    comps = np.asarray(city_component_list[city_idx], dtype=np.float32)
    n_comp = comps.shape[0]
    n_rows = 1 + n_comp
    fig_h = 1.45 * n_rows + 0.9
    fig, axes = plt.subplots(n_rows, 1, figsize=(8.6, fig_h), sharex=True, gridspec_kw={'hspace': 0.14})
    _draw_pretty_line(axes[0], dates, signal, color=CEEMDAN_COLORS['signal'], label_text='Original AQI', lw=1.8, fill_alpha=0.1)
    _style_ceemdan_axis(axes[0], show_x=False)
    for k in range(n_comp):
        is_last_residual = bool(USE_RESIDUAL_COMPONENT and k == n_comp - 1)
        if is_last_residual:
            comp_label = 'Residual / Trend'
            comp_color = CEEMDAN_COLORS['residual']
            lw = 1.6
            fill_alpha = 0.1
        else:
            comp_label = f'IMF {k + 1}'
            comp_color = IMF_PASTEL_COLORS[k % len(IMF_PASTEL_COLORS)]
            lw = 1.35
            fill_alpha = 0.09
        _draw_pretty_line(axes[k + 1], dates, comps[k], color=comp_color, label_text=comp_label, lw=lw, fill_alpha=fill_alpha)
        _style_ceemdan_axis(axes[k + 1], show_x=False)
    fig.suptitle(f'{city_name_en} - CEEMDAN Decomposition', fontsize=16, color=CEEMDAN_COLORS['text'], y=0.995)
    axes[-1].set_xlabel('Date', color=CEEMDAN_COLORS['text'])
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    file_base = f'ceemdan_{safe_filename(city_name_en)}'
    fig.savefig(os.path.join(CEEMDAN_PLOT_DIR, f'{file_base}.png'), bbox_inches='tight', dpi=320)
    fig.savefig(os.path.join(CEEMDAN_PLOT_DIR, f'{file_base}.pdf'), bbox_inches='tight')
    plt.show(fig)
for city_idx, city in enumerate(city_order):
    plot_city_ceemdan_from_current_script(city, city_idx)
print('各市 CEEMDAN 美化图已保存到：', CEEMDAN_PLOT_DIR)


In [ ]:
CITY_NAME_EN = {'长春': 'Changchun', '吉林': 'Jilin', '四平': 'Siping', '辽源': 'Liaoyuan', '通化': 'Tonghua', '白山': 'Baishan', '松原': 'Songyuan', '白城': 'Baicheng', '延边': 'Yanbian'}

def city_en(name):
    return CITY_NAME_EN.get(name, name)

def city_list_en(city_list):
    return [city_en(c) for c in city_list]

def rename_city_df(df):
    df2 = df.copy()
    if isinstance(df2.index, pd.Index):
        df2.index = [city_en(x) for x in df2.index]
    if isinstance(df2.columns, pd.Index):
        df2.columns = [city_en(x) for x in df2.columns]
    return df2
city_order_en = city_list_en(city_order) if 'city_order' in globals() else None


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
VIS_SWITCH = {'data_overview': True, 'feature_selection': True, 'graph_structure': True, 'ceemdan_branch': True, 'prediction_fit': True, 'model_metrics': True, 'ablation': True, 'fusion': True, 'consistency': True, 'grouped_metric_excel': True}
FORCE_ADJ_VIS = True
VIS_DIR = os.path.join(SAVE_DIR, 'all_visualizations_en')
for sub in ['00_data_overview', '01_feature_selection', '02_graph_structure', '03_ceemdan_branch', '04_prediction_fit', '05_model_metrics', '06_ablation', '07_fusion', '08_consistency']:
    ensure_dir(os.path.join(VIS_DIR, sub))
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = '#FBFEFF'
plt.rcParams['axes.facecolor'] = '#FBFEFF'
plt.rcParams['axes.edgecolor'] = '#C9DCEB'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#E6F1F7'
plt.rcParams['grid.alpha'] = 0.75
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
SOFT_COLORS = ['#8FD3D1', '#A8D88C', '#F6C77A', '#F3A6B7', '#92A8E8', '#C7B5EA', '#A7E0C2', '#9BD4F5']
MODEL_COLOR_MAP = {'DualBranch-CEGNetStyle': '#8FD3D1', 'Fusion': '#92A8E8', 'Persistence': '#F6C77A', 'MA7': '#F3A6B7'}
ABLATION_COLOR_MAP = {'Full': '#8FD3D1', 'w/o CEEMDAN Branch': '#F3A6B7', 'w/o Exogenous Branch': '#F6C77A', 'w/o Graph Convolution': '#92A8E8', 'w/o Gated Fusion': '#C7B5EA', 'w/o Skip Connection': '#A7E0C2', 'w/o Event Loss': '#9BD4F5'}
THR_COLOR_MAP = {100: '#F6C77A', 150: '#F3A6B7', 200: '#C7B5EA'}

def make_cmap(colors, name):
    return LinearSegmentedColormap.from_list(name, colors)
CMAP_FRESH = make_cmap(['#F8FCFF', '#E3F7F6', '#BCE9D8', '#8FD3D1', '#92A8E8'], 'fresh')
CMAP_WARM = make_cmap(['#FFFDF8', '#FCEFD4', '#F6D8B5', '#F3A6B7', '#C7B5EA'], 'warm')
CMAP_MINT = make_cmap(['#FCFFFD', '#EAF9F0', '#CFEFD9', '#A8D88C', '#8FD3D1'], 'mint')

def save_fig(fig, path):
    ensure_dir(os.path.dirname(path))
    fig.tight_layout()
    fig.savefig(path, dpi=260, bbox_inches='tight')
    plt.show(fig)

def vis_try(tag, func):
    try:
        func()
        print(f'[VIS OK] {tag}')
    except Exception as e:
        print(f'[VIS FAIL] {tag}: {e}')

def heatmap_plot(df, title, save_path, cmap=CMAP_FRESH, annot=False, fmt='.3f', vmin=None, vmax=None):
    fig_w = max(7, 0.7 * df.shape[1] + 3)
    fig_h = max(4, 0.45 * df.shape[0] + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(df.values, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(df.shape[1]))
    ax.set_xticklabels(df.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(df.shape[0]))
    ax.set_yticklabels(df.index)
    ax.set_title(title)
    if annot and df.shape[0] * df.shape[1] <= 180:
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                val = df.iat[i, j]
                txt = 'NA' if pd.isna(val) else format(val, fmt)
                ax.text(j, i, txt, ha='center', va='center', fontsize=8, color='#38556B')
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.outline.set_edgecolor('#C9DCEB')
    save_fig(fig, save_path)

def lollipop_plot(labels, values, title, save_path, color='#8FD3D1', xlabel='Value', sort_desc=True):
    ser = pd.Series(values, index=labels)
    ser = ser.sort_values(ascending=not sort_desc)
    fig, ax = plt.subplots(figsize=(9, max(4.5, 0.35 * len(ser) + 1.8)))
    y = np.arange(len(ser))
    ax.hlines(y=y, xmin=0, xmax=ser.values, color='#DCEEF6', linewidth=3)
    ax.scatter(ser.values, y, s=90, color=color, edgecolor='white', linewidth=1.5, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(ser.index)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    for yi, v in zip(y, ser.values):
        ax.text(v, yi, f' {v:.4f}', va='center', ha='left', fontsize=9, color='#466178')
    save_fig(fig, save_path)

def polar_bar_plot(labels, values, title, save_path):
    values = np.asarray(values, dtype=float)
    labels = list(labels)
    n = len(labels)
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    width = 2 * np.pi / n * 0.86
    colors = [SOFT_COLORS[i % len(SOFT_COLORS)] for i in range(n)]
    fig = plt.figure(figsize=(9, 9))
    ax = plt.subplot(111, polar=True)
    ax.bar(theta, values, width=width, bottom=0.0, color=colors, edgecolor='white', linewidth=1.5, alpha=0.95)
    ax.set_xticks(theta)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title, pad=22)
    ax.grid(color='#E6F1F7', linestyle='--', alpha=0.8)
    for ang, val in zip(theta, values):
        ax.text(ang, val + max(values) * 0.05 + 1e-08, f'{val:.2f}', ha='center', va='center', fontsize=8, color='#4A6175')
    save_fig(fig, save_path)

def bubble_matrix_plot(df, title, save_path, cmap=CMAP_FRESH):
    arr = df.values.astype(float)
    nrow, ncol = arr.shape
    xv, yv = np.meshgrid(np.arange(ncol), np.arange(nrow))
    flat = arr.reshape(-1)
    abs_flat = np.abs(flat)
    s_min, s_max = (40, 900)
    if np.nanmax(abs_flat) - np.nanmin(abs_flat) < 1e-10:
        sizes = np.full_like(abs_flat, (s_min + s_max) / 2.0)
    else:
        sizes = s_min + (abs_flat - np.nanmin(abs_flat)) / (np.nanmax(abs_flat) - np.nanmin(abs_flat) + 1e-08) * (s_max - s_min)
    fig_w = max(7, 0.65 * ncol + 3)
    fig_h = max(5, 0.55 * nrow + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sc = ax.scatter(xv.reshape(-1), yv.reshape(-1), s=sizes, c=flat, cmap=cmap, alpha=0.9, edgecolors='white', linewidths=1.0)
    ax.set_xticks(np.arange(ncol))
    ax.set_xticklabels(df.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(nrow))
    ax.set_yticklabels(df.index)
    ax.invert_yaxis()
    ax.set_title(title)
    cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
    cb.outline.set_edgecolor('#C9DCEB')
    save_fig(fig, save_path)

def radar_plot(score_df, title, save_path):
    labels = score_df.columns.tolist()
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
    angles = np.concatenate([angles, [angles[0]]])
    fig = plt.figure(figsize=(8.5, 8.5))
    ax = plt.subplot(111, polar=True)
    for i, (name, row) in enumerate(score_df.iterrows()):
        vals = row.values.astype(float).tolist()
        vals = vals + [vals[0]]
        color = MODEL_COLOR_MAP.get(name, SOFT_COLORS[i % len(SOFT_COLORS)])
        ax.plot(angles, vals, linewidth=2.2, color=color, label=name)
        ax.fill(angles, vals, color=color, alpha=0.16)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels([])
    ax.set_title(title, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.12), frameon=False)
    save_fig(fig, save_path)

def reliability_curve_from_aqi(y_true, y_pred, thr, n_bins=10):
    y_true_bin = (np.asarray(y_true).reshape(-1) >= thr).astype(int)
    prob = event_prob_from_aqi(np.asarray(y_pred).reshape(-1), thr=thr)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    xs, ys = ([], [])
    for i in range(n_bins):
        if i < n_bins - 1:
            mask = (prob >= bins[i]) & (prob < bins[i + 1])
        else:
            mask = (prob >= bins[i]) & (prob <= bins[i + 1])
        if mask.sum() == 0:
            continue
        xs.append(prob[mask].mean())
        ys.append(y_true_bin[mask].mean())
    return (np.array(xs), np.array(ys))

def dumbbell_plot(df, left_col, right_col, title, save_path, left_color='#8FD3D1', right_color='#92A8E8'):
    df = df.copy().sort_values(right_col)
    fig, ax = plt.subplots(figsize=(9, max(4.5, 0.45 * len(df) + 1.5)))
    y = np.arange(len(df))
    ax.hlines(y, df[left_col], df[right_col], color='#D8EAF3', linewidth=2.5)
    ax.scatter(df[left_col], y, s=90, color=left_color, edgecolor='white', linewidth=1.3, label=left_col, zorder=3)
    ax.scatter(df[right_col], y, s=90, color=right_color, edgecolor='white', linewidth=1.3, label=right_col, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(df.index.tolist())
    ax.set_title(title)
    ax.legend(frameon=False)
    for yi, lv, rv in zip(y, df[left_col].values, df[right_col].values):
        ax.text(lv, yi, f'{lv:.2f} ', ha='right', va='center', fontsize=8, color='#466178')
        ax.text(rv, yi, f' {rv:.2f}', ha='left', va='center', fontsize=8, color='#466178')
    save_fig(fig, save_path)

def normalize_for_radar(df, lower_better_cols):
    out = df.copy().astype(float)
    for col in out.columns:
        arr = out[col].values.astype(float)
        mn, mx = (np.nanmin(arr), np.nanmax(arr))
        if mx - mn < 1e-12:
            out[col] = 1.0
        elif col in lower_better_cols:
            out[col] = (mx - arr) / (mx - mn + 1e-08)
        else:
            out[col] = (arr - mn) / (mx - mn + 1e-08)
    return out
city_order_en = city_list_en(city_order)
if USE_RESIDUAL_COMPONENT and max_comp_total >= 1:
    component_names_full = [f'IMF{i + 1}' for i in range(max_comp_total - 1)] + ['Residual']
else:
    component_names_full = [f'IMF{i + 1}' for i in range(max_comp_total)]

def make_data_overview_plots():
    fig, ax = plt.subplots(figsize=(12.5, 2.2))
    ax.axvspan(pd.to_datetime(train_dates[0]), pd.to_datetime(train_dates[-1]), color='#E8F7F5', alpha=0.95, label='Train')
    ax.axvspan(pd.to_datetime(val_dates[0]), pd.to_datetime(val_dates[-1]), color='#FFF2D8', alpha=0.95, label='Validation')
    ax.axvspan(pd.to_datetime(test_dates[0]), pd.to_datetime(test_dates[-1]), color='#FBE3EA', alpha=0.95, label='Test')
    ax.set_xlim(pd.to_datetime(dates[0]), pd.to_datetime(dates[-1]))
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_title('Temporal Split Overview')
    ax.legend(frameon=False, loc='upper left', ncol=3)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    save_fig(fig, os.path.join(VIS_DIR, '00_data_overview', '01_time_split_timeline.png'))
    tmp = df_model[[DATE_COL, CITY_COL, TARGET_COL]].copy()
    tmp['Month'] = tmp[DATE_COL].dt.to_period('M').astype(str)
    monthly_heat = tmp.groupby(['Month', CITY_COL])[TARGET_COL].mean().unstack().reindex(columns=city_order)
    monthly_heat.columns = city_order_en
    heatmap_plot(monthly_heat.T, 'Monthly Mean AQI by City', os.path.join(VIS_DIR, '00_data_overview', '02_city_monthly_aqi_heatmap.png'), cmap=CMAP_WARM, annot=False)
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    data = [Y_train_raw.reshape(-1), Y_val_raw.reshape(-1), Y_test_raw.reshape(-1)]
    parts = ax.violinplot(data, positions=[1, 2, 3], showmeans=False, showmedians=True, showextrema=False)
    for body, c in zip(parts['bodies'], ['#8FD3D1', '#F6C77A', '#F3A6B7']):
        body.set_facecolor(c)
        body.set_edgecolor('white')
        body.set_alpha(0.92)
    parts['cmedians'].set_color('#4C6579')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Train', 'Validation', 'Test'])
    ax.set_ylabel('AQI')
    ax.set_title('AQI Distribution across Data Splits')
    save_fig(fig, os.path.join(VIS_DIR, '00_data_overview', '03_aqi_violin_by_split.png'))

def make_feature_selection_plots():
    show_df = mrmr_rank_df.head(min(20, len(mrmr_rank_df))).copy()
    lollipop_plot(labels=show_df['Feature'].tolist(), values=show_df['mRMR_Score'].tolist(), title='Top Features ranked by mRMR', save_path=os.path.join(VIS_DIR, '01_feature_selection', '01_mrmr_lollipop.png'), color='#8FD3D1', xlabel='mRMR Score', sort_desc=True)
    stable_show = stability_df.head(min(20, len(stability_df))).copy()
    polar_bar_plot(labels=stable_show['Feature'].tolist(), values=stable_show['Stability_Freq'].tolist(), title='Feature Stability Frequency', save_path=os.path.join(VIS_DIR, '01_feature_selection', '02_stability_polar_bar.png'))
    corr_df = train_df_fs[selected_features].corr().round(3)
    heatmap_plot(corr_df, 'Correlation Heatmap of Selected Features', os.path.join(VIS_DIR, '01_feature_selection', '03_selected_feature_corr_heatmap.png'), cmap=CMAP_FRESH, annot=False)
    source_map = selection_source_df.copy()
    source_map['stable_flag'] = source_map['Source'].str.contains('stable').astype(int)
    source_map['mandatory_flag'] = source_map['Source'].str.contains('mandatory').astype(int)
    source_plot = source_map.set_index('Feature')[['stable_flag', 'mandatory_flag']]
    source_plot.columns = ['Stable', 'Mandatory']
    heatmap_plot(source_plot, 'Selection Source of Final Features', os.path.join(VIS_DIR, '01_feature_selection', '04_feature_source_heatmap.png'), cmap=CMAP_MINT, annot=True, fmt='.0f', vmin=0, vmax=1)

def make_graph_structure_plots():
    dist_df = pd.DataFrame(dist_mat, index=city_order_en, columns=city_order_en)
    heatmap_plot(dist_df, 'Geographical Distance Matrix (km)', os.path.join(VIS_DIR, '02_graph_structure', '01_distance_heatmap.png'), cmap=CMAP_FRESH, annot=True, fmt='.1f')
    A_raw_df = pd.DataFrame(A_static_raw, index=city_order_en, columns=city_order_en)
    heatmap_plot(A_raw_df, 'Raw Adjacency Matrix', os.path.join(VIS_DIR, '02_graph_structure', '02_adjacency_raw_heatmap.png'), cmap=CMAP_MINT, annot=True, fmt='.3f')
    A_rw_df = pd.DataFrame(A_static_rw, index=city_order_en, columns=city_order_en)
    heatmap_plot(A_rw_df, 'Row-normalized Adjacency Matrix', os.path.join(VIS_DIR, '02_graph_structure', '03_adjacency_rw_heatmap.png'), cmap=CMAP_FRESH, annot=True, fmt='.3f')
    bubble_matrix_plot(A_raw_df, 'Adjacency Matrix Bubble Plot', os.path.join(VIS_DIR, '02_graph_structure', '04_adjacency_bubble_matrix.png'), cmap=CMAP_WARM)
    fig, ax = plt.subplots(figsize=(9, 7))
    max_w = np.max(A_static_raw) if np.max(A_static_raw) > 0 else 1.0
    for i, ci in enumerate(city_order):
        xi, yi = CITY_COORDS[ci]
        for j, cj in enumerate(city_order):
            if i >= j:
                continue
            w = max(A_static_raw[i, j], A_static_raw[j, i])
            if w > 0:
                xj, yj = CITY_COORDS[cj]
                ax.plot([xi, xj], [yi, yj], color='#92A8E8', linewidth=0.8 + 4.2 * w / max_w, alpha=0.32)
    node_sizes = 600 + 2500 * (w_space / (w_space.max() + 1e-08))
    xs = [CITY_COORDS[c][0] for c in city_order]
    ys = [CITY_COORDS[c][1] for c in city_order]
    ax.scatter(xs, ys, s=node_sizes, c=w_space, cmap=CMAP_MINT, edgecolor='white', linewidth=1.8, zorder=3)
    for c in city_order:
        x, y = CITY_COORDS[c]
        ax.text(x, y + 0.08, city_en(c), ha='center', va='bottom', fontsize=10, color='#425C73')
    ax.set_title('Static Spatial Graph of Jilin Province')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    save_fig(fig, os.path.join(VIS_DIR, '02_graph_structure', '05_geo_graph_network.png'))
    weight_df_en = weight_df.copy()
    weight_df_en['City'] = weight_df_en['City'].map(city_en)
    lollipop_plot(labels=weight_df_en['City'].tolist(), values=weight_df_en['Weight_Final'].tolist(), title='Provincial Spatial Weights', save_path=os.path.join(VIS_DIR, '02_graph_structure', '06_province_spatial_weight_lollipop.png'), color='#92A8E8', xlabel='Weight', sort_desc=True)

def make_ceemdan_branch_plots():
    comp_cnt_df = component_summary_df.copy()
    comp_cnt_df['City'] = comp_cnt_df['City'].map(city_en)
    comp_cnt_df = comp_cnt_df.set_index('City')[['Train_IMF_Count', 'Used_IMF_Count', 'Total_Used_Components']]
    heatmap_plot(comp_cnt_df, 'CEEMDAN Component Count Summary', os.path.join(VIS_DIR, '03_ceemdan_branch', '01_component_count_heatmap.png'), cmap=CMAP_MINT, annot=True, fmt='.0f')
    comp_energy = (component_tensor[:train_end] ** 2).mean(axis=0)
    comp_energy_df = pd.DataFrame(comp_energy, index=city_order_en, columns=component_names_full)
    heatmap_plot(comp_energy_df, 'Component Energy Heatmap (Train)', os.path.join(VIS_DIR, '03_ceemdan_branch', '02_component_energy_heatmap_train.png'), cmap=CMAP_WARM, annot=False)
    energy_rows = []
    for city_idx, city in enumerate(city_order):
        series = pivot_aqi.loc[dates, city].values.astype(np.float32)
        comps = city_component_list[city_idx]
        if USE_RESIDUAL_COMPONENT and comps.shape[0] >= 1:
            comp_names = [f'IMF{i + 1}' for i in range(comps.shape[0] - 1)] + ['Residual']
        else:
            comp_names = [f'IMF{i + 1}' for i in range(comps.shape[0])]
        recon = comps.sum(axis=0)
        recon_rmse = calc_metrics(series, recon)['RMSE']
        nrows = comps.shape[0] + 2
        fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(13, 2.2 * nrows), sharex=True)
        dt = pd.to_datetime(dates)
        axes[0].plot(dt, series, color='#4D687D', linewidth=2.0)
        axes[0].fill_between(dt, series, np.zeros_like(series), color='#DDEFF5', alpha=0.45)
        axes[0].set_title(f'{city_en(city)} - Original AQI')
        for k in range(comps.shape[0]):
            axes[k + 1].plot(dt, comps[k], color=SOFT_COLORS[k % len(SOFT_COLORS)], linewidth=1.6)
            axes[k + 1].axhline(0, color='#D8EAF3', linewidth=1)
            axes[k + 1].set_ylabel(comp_names[k], rotation=0, labelpad=28, va='center')
        axes[-1].plot(dt, series, color='#4D687D', linewidth=2.0, label='Original')
        axes[-1].plot(dt, recon, color='#8FD3D1', linewidth=1.8, linestyle='--', label='Sum of kept components')
        axes[-1].set_title(f'Reconstruction using kept components | RMSE={recon_rmse:.3f}')
        axes[-1].legend(frameon=False, ncol=2, loc='upper right')
        axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        save_fig(fig, os.path.join(VIS_DIR, '03_ceemdan_branch', f'03_ceemdan_decompose_{safe_filename(city_en(city))}.png'))
        train_series_energy = pivot_aqi.loc[train_dates, city].values.astype(np.float32)
        imfs_train_energy, resid_train_energy = decompose_series_ceemdan(train_series_energy)
        full_energy = float(np.sum(train_series_energy ** 2) + 1e-08)
        kept_energy = float(np.sum(imfs_train_energy[:min(MAX_IMF_KEEP, imfs_train_energy.shape[0])] ** 2) + np.sum(resid_train_energy ** 2))
        energy_rows.append({'City': city_en(city), 'Retained_Energy_Ratio': kept_energy / (full_energy + 1e-08), 'Recon_RMSE': recon_rmse})
    energy_ratio_df = pd.DataFrame(energy_rows).sort_values('Retained_Energy_Ratio', ascending=False).reset_index(drop=True)
    energy_ratio_df.to_excel(os.path.join(SAVE_DIR, 'ceemdan_energy_retention_en.xlsx'), index=False)
    lollipop_plot(labels=energy_ratio_df['City'].tolist(), values=energy_ratio_df['Retained_Energy_Ratio'].tolist(), title='Retained Energy Ratio of CEEMDAN Components', save_path=os.path.join(VIS_DIR, '03_ceemdan_branch', '04_retained_energy_ratio_lollipop.png'), color='#A8D88C', xlabel='Retained Energy Ratio', sort_desc=True)
    lollipop_plot(labels=energy_ratio_df.sort_values('Recon_RMSE', ascending=True)['City'].tolist(), values=energy_ratio_df.sort_values('Recon_RMSE', ascending=True)['Recon_RMSE'].tolist(), title='Reconstruction Error of Kept CEEMDAN Components', save_path=os.path.join(VIS_DIR, '03_ceemdan_branch', '05_reconstruction_rmse_lollipop.png'), color='#F3A6B7', xlabel='RMSE', sort_desc=False)

def make_prediction_fit_plots():

    def province_plot(date_seq, y_true, pred_dict, phase_name, save_name):
        fig, ax = plt.subplots(figsize=(13.5, 5.2))
        dt = pd.to_datetime(date_seq)
        ax.plot(dt, province_series(y_true), color='#4D687D', linewidth=2.5, label='True Provincial AQI')
        for model_name, pred in pred_dict.items():
            ax.plot(dt, province_series(pred), linewidth=2.0, color=MODEL_COLOR_MAP.get(model_name, '#92A8E8'), alpha=0.95, label=model_name)
        for thr, c in THR_COLOR_MAP.items():
            ax.axhline(thr, color=c, linestyle='--', linewidth=1.2, alpha=0.8)
        ax.set_title(f'Provincial AQI Fit Comparison ({phase_name})')
        ax.legend(frameon=False, ncol=2)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        save_fig(fig, os.path.join(VIS_DIR, '04_prediction_fit', save_name))
    province_plot(dates_val_final, Y_val_raw, {k: v['val'] for k, v in model_dict_eval.items()}, 'Validation', '01_province_fit_validation.png')
    province_plot(dates_test_final, Y_test_raw, {k: v['test'] for k, v in model_dict_eval.items()}, 'Test', '02_province_fit_test.png')
    dt_all = pd.to_datetime(list(dates_val_final) + list(dates_test_final))
    split_point = pd.to_datetime(dates_test_final[0])
    for i, city in enumerate(city_order):
        y_true = np.concatenate([Y_val_raw[:, i], Y_test_raw[:, i]])
        y_main = np.concatenate([pred_val[:, i], pred_test[:, i]])
        y_fusion = np.concatenate([fused_val[:, i], fused_test[:, i]])
        fig, ax = plt.subplots(figsize=(13, 4.8))
        ax.axvspan(dt_all[0], pd.to_datetime(dates_val_final[-1]), color='#FFF6E5', alpha=0.55, label='Validation')
        ax.axvspan(split_point, dt_all[-1], color='#FDEBF1', alpha=0.45, label='Test')
        ax.plot(dt_all, y_true, color='#4D687D', linewidth=2.3, label='True AQI')
        ax.plot(dt_all, y_main, color='#8FD3D1', linewidth=1.9, label='DualBranch-CEGNetStyle')
        ax.plot(dt_all, y_fusion, color='#92A8E8', linewidth=1.9, label='Fusion')
        event_mask = y_true >= 100
        ax.fill_between(dt_all, 0, y_true, where=event_mask, color='#F6C77A', alpha=0.14)
        for thr, c in THR_COLOR_MAP.items():
            ax.axhline(thr, color=c, linestyle='--', linewidth=1.0, alpha=0.75)
        ax.set_title(f'{city_en(city)} AQI Fit Comparison (Validation + Test)')
        ax.legend(frameon=False, ncol=4, loc='upper left')
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        save_fig(fig, os.path.join(VIS_DIR, '04_prediction_fit', f'03_city_fit_{safe_filename(city_en(city))}.png'))
    for model_name, pack in model_dict_eval.items():
        yt = Y_test_raw.reshape(-1)
        yp = pack['test'].reshape(-1)
        fig, ax = plt.subplots(figsize=(6.8, 5.8))
        hb = ax.hexbin(yt, yp, gridsize=28, cmap=CMAP_FRESH, mincnt=1, linewidths=0.15)
        mx = max(np.max(yt), np.max(yp)) + 5
        ax.plot([0, mx], [0, mx], linestyle='--', linewidth=1.3, color='#F3A6B7')
        met = calc_extended_metrics(yt, yp, thresholds=EVENT_THRESHOLDS)
        ax.set_title(f'{model_name} | Observed vs Predicted Density (Test)')
        ax.set_xlabel('Observed AQI')
        ax.set_ylabel('Predicted AQI')
        ax.text(0.03, 0.97, f"RMSE={met['RMSE']:.2f}\nMAE={met['MAE']:.2f}\nR2={met['R2']:.3f}", transform=ax.transAxes, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.35', facecolor='#FFFFFF', edgecolor='#D6EAF4'))
        cb = fig.colorbar(hb, ax=ax, fraction=0.045, pad=0.03)
        cb.outline.set_edgecolor('#C9DCEB')
        save_fig(fig, os.path.join(VIS_DIR, '04_prediction_fit', f'04_hexbin_{safe_filename(model_name)}.png'))
    fig, ax = plt.subplots(figsize=(8.8, 5.5))
    residual_data = [model_dict_eval[m]['test'].reshape(-1) - Y_test_raw.reshape(-1) for m in model_dict_eval.keys()]
    parts = ax.violinplot(residual_data, positions=np.arange(1, len(residual_data) + 1), showmedians=True, showextrema=False)
    for body, name in zip(parts['bodies'], model_dict_eval.keys()):
        body.set_facecolor(MODEL_COLOR_MAP.get(name, '#8FD3D1'))
        body.set_edgecolor('white')
        body.set_alpha(0.92)
    parts['cmedians'].set_color('#4D687D')
    ax.axhline(0, color='#F3A6B7', linestyle='--', linewidth=1.2)
    ax.set_xticks(np.arange(1, len(residual_data) + 1))
    ax.set_xticklabels(list(model_dict_eval.keys()), rotation=15)
    ax.set_ylabel('Residual = Predicted - True')
    ax.set_title('Residual Distribution on Test Set')
    save_fig(fig, os.path.join(VIS_DIR, '04_prediction_fit', '05_test_residual_violin.png'))
    for thr in EVENT_THRESHOLDS:
        fig, ax = plt.subplots(figsize=(6.8, 5.8))
        ax.plot([0, 1], [0, 1], linestyle='--', color='#C7B5EA', linewidth=1.2)
        for model_name, pack in model_dict_eval.items():
            x, y = reliability_curve_from_aqi(Y_test_raw, pack['test'], thr=thr, n_bins=10)
            if len(x) == 0:
                continue
            ax.plot(x, y, marker='o', linewidth=2, markersize=5, color=MODEL_COLOR_MAP.get(model_name, '#8FD3D1'), label=model_name)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Observed Frequency')
        ax.set_title(f'Reliability Curve (AQI ≥ {thr})')
        ax.legend(frameon=False)
        save_fig(fig, os.path.join(VIS_DIR, '04_prediction_fit', f'06_reliability_thr_{thr}.png'))

def make_model_metric_plots():
    metric_cols_show = ['RMSE', 'MAE', 'R2', 'SMAPE', 'EHR_100', 'CSI_100', 'AUC_100', 'EHR_150', 'AUC_150', 'EHR_200', 'AUC_200']
    overall_test = overall_metrics_df[overall_metrics_df['Phase'] == 'Test'].set_index('Model')[metric_cols_show]
    heatmap_plot(overall_test, 'Overall Test Metrics', os.path.join(VIS_DIR, '05_model_metrics', '01_overall_test_metric_heatmap.png'), cmap=CMAP_FRESH, annot=True, fmt='.3f')
    province_test = province_metrics_df[province_metrics_df['Phase'] == 'Test'].set_index('Model')[metric_cols_show]
    heatmap_plot(province_test, 'Provincial Test Metrics', os.path.join(VIS_DIR, '05_model_metrics', '02_province_test_metric_heatmap.png'), cmap=CMAP_MINT, annot=True, fmt='.3f')
    radar_src = overall_test[['RMSE', 'MAE', 'R2', 'EHR_100', 'CSI_100', 'AUC_100']].copy()
    radar_score = normalize_for_radar(radar_src, lower_better_cols=['RMSE', 'MAE'])
    radar_plot(radar_score, 'Comprehensive Model Comparison (Test)', os.path.join(VIS_DIR, '05_model_metrics', '03_model_radar_test.png'))
    city_test_full = city_metrics_df[(city_metrics_df['Model'] == 'DualBranch-CEGNetStyle') & (city_metrics_df['Phase'] == 'Test')].copy()
    city_test_full['City'] = city_test_full['City'].map(city_en)
    city_test_full = city_test_full.set_index('City')[['RMSE', 'MAE', 'R2', 'EHR_100', 'CSI_100', 'AUC_100']]
    heatmap_plot(city_test_full, 'City-level Metrics of DualBranch-CEGNetStyle (Test)', os.path.join(VIS_DIR, '05_model_metrics', '04_full_city_metric_heatmap_test.png'), cmap=CMAP_WARM, annot=True, fmt='.3f')
    rank_targets = {'RMSE': False, 'R2': True, 'EHR_100': True, 'AUC_100': True}
    for metric, desc in rank_targets.items():
        ser = overall_test[metric].sort_values(ascending=not desc)
        lollipop_plot(labels=ser.index.tolist(), values=ser.values.tolist(), title=f'Model Ranking by {metric} (Test)', save_path=os.path.join(VIS_DIR, '05_model_metrics', f'05_rank_{metric}.png'), color='#92A8E8' if desc else '#F3A6B7', xlabel=metric, sort_desc=desc)

def make_ablation_plots():
    abl_cols = ['RMSE', 'MAE', 'R2', 'SMAPE', 'EHR_100', 'CSI_100', 'AUC_100', 'EHR_150', 'AUC_150', 'EHR_200', 'AUC_200']
    abl_test = ablation_overall_df[ablation_overall_df['Phase'] == 'Test'].set_index('Variant')[abl_cols]
    abl_test = abl_test.reindex(variants_to_run)
    heatmap_plot(abl_test, 'Ablation Metrics on Test Set', os.path.join(VIS_DIR, '06_ablation', '01_ablation_overall_heatmap_test.png'), cmap=CMAP_WARM, annot=True, fmt='.3f')
    full_row = abl_test.loc['Full']
    delta_df = abl_test.subtract(full_row, axis=1).drop(index='Full')
    heatmap_plot(delta_df[['RMSE', 'MAE', 'R2', 'EHR_100', 'CSI_100', 'AUC_100']], 'Delta Relative to Full Model (Test)', os.path.join(VIS_DIR, '06_ablation', '02_ablation_delta_vs_full_heatmap.png'), cmap=CMAP_FRESH, annot=True, fmt='.3f')
    ceemdan_compare = abl_test.loc[['Full', 'w/o CEEMDAN Branch'], ['RMSE', 'MAE', 'R2', 'EHR_100', 'CSI_100', 'AUC_100']].copy()
    heatmap_plot(ceemdan_compare, 'Contribution of the CEEMDAN Branch', os.path.join(VIS_DIR, '06_ablation', '03_ceemdan_contribution_heatmap.png'), cmap=CMAP_MINT, annot=True, fmt='.3f')
    abl_city_rmse = ablation_city_df[ablation_city_df['Phase'] == 'Test'].copy()
    abl_city_rmse['City'] = abl_city_rmse['City'].map(city_en)
    abl_city_rmse = abl_city_rmse.pivot(index='Variant', columns='City', values='RMSE')
    abl_city_rmse = abl_city_rmse.reindex(index=variants_to_run, columns=city_order_en)
    heatmap_plot(abl_city_rmse, 'City-level RMSE in Ablation Study (Test)', os.path.join(VIS_DIR, '06_ablation', '04_ablation_city_rmse_heatmap_test.png'), cmap=CMAP_WARM, annot=True, fmt='.3f')

def make_fusion_plots():
    coef_cols = [c for c in fusion_coef_df.columns if c not in ['City', 'Intercept']]
    fusion_coef_heat = fusion_coef_df.copy()
    fusion_coef_heat['City'] = fusion_coef_heat['City'].map(city_en)
    fusion_coef_heat = fusion_coef_heat.set_index('City')[coef_cols]
    heatmap_plot(fusion_coef_heat, 'City-wise Fusion Coefficients', os.path.join(VIS_DIR, '07_fusion', '01_fusion_coefficient_heatmap.png'), cmap=CMAP_FRESH, annot=True, fmt='.3f')
    bias_compare = pd.DataFrame({'MainModel_Bias': main_bias_df.set_index('City')['Bias'], 'Fusion_Bias': fusion_bias_df.set_index('City')['Bias']})
    bias_compare.index = [city_en(x) for x in bias_compare.index]
    dumbbell_plot(bias_compare, 'MainModel_Bias', 'Fusion_Bias', 'City-wise Bias Correction Comparison', os.path.join(VIS_DIR, '07_fusion', '02_bias_dumbbell.png'), left_color='#8FD3D1', right_color='#92A8E8')

def make_consistency_plots():
    cons_cols = ['Pred_vs_TrueAQI_RMSE', 'Pred_vs_TrueAQI_MAE', 'Pred_vs_RebuildTrueAQI_RMSE', 'Pred_vs_RebuildTrueAQI_MAE', 'TrueAQI_vs_RebuildTrueAQI_RMSE', 'TrueAQI_vs_RebuildTrueAQI_MAE', 'AQI_Grade_Acc']
    cons_test = consistency_df[consistency_df['Phase'] == 'Test'].set_index('Model')[cons_cols]
    heatmap_plot(cons_test, 'Pollutant Consistency Metrics (Test)', os.path.join(VIS_DIR, '08_consistency', '01_consistency_heatmap_test.png'), cmap=CMAP_MINT, annot=True, fmt='.3f')
    pp_rmse = primary_pollutant_df[primary_pollutant_df['Phase'] == 'Test'].pivot(index='Model', columns='PrimaryPollutant', values='RMSE_TrueAQI')
    pp_rmse = pp_rmse.reindex(index=list(model_dict_eval.keys()))
    heatmap_plot(pp_rmse, 'RMSE by Primary Pollutant Group (Test)', os.path.join(VIS_DIR, '08_consistency', '02_primary_pollutant_rmse_heatmap_test.png'), cmap=CMAP_WARM, annot=True, fmt='.3f')
    pp_acc = primary_pollutant_df[primary_pollutant_df['Phase'] == 'Test'].pivot(index='Model', columns='PrimaryPollutant', values='AQI_Grade_Acc')
    pp_acc = pp_acc.reindex(index=list(model_dict_eval.keys()))
    heatmap_plot(pp_acc, 'AQI Grade Accuracy by Primary Pollutant Group (Test)', os.path.join(VIS_DIR, '08_consistency', '03_primary_pollutant_gradeacc_heatmap_test.png'), cmap=CMAP_FRESH, annot=True, fmt='.3f')

def export_grouped_metric_excel():
    out_path = os.path.join(VIS_DIR, 'grouped_metric_tables_en.xlsx')
    with pd.ExcelWriter(out_path) as writer:
        save_grouped_sheets(df=overall_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='overall', sort_cols=['Phase', 'RMSE'])
        save_grouped_sheets(df=province_metrics_df, id_cols=['Model', 'Phase', 'City'], groups=METRIC_GROUPS, writer=writer, prefix='province', sort_cols=['Phase', 'RMSE'])
        save_grouped_sheets(df=consistency_df, id_cols=['Model', 'Phase'], groups=CONSISTENCY_GROUPS, writer=writer, prefix='consistency', sort_cols=['Phase', 'Pred_vs_RebuildTrueAQI_RMSE'])
        save_grouped_sheets(df=primary_pollutant_df, id_cols=['Model', 'Phase', 'PrimaryPollutant'], groups=PRIMARY_POLLUTANT_GROUPS, writer=writer, prefix='primary_pollutant', sort_cols=['Phase', 'PrimaryPollutant', 'RMSE_TrueAQI'])
        save_grouped_sheets(df=ablation_overall_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, writer=writer, prefix='ablation_overall', sort_cols=['Phase', 'RMSE'])
        save_grouped_sheets(df=ablation_province_df, id_cols=['Variant', 'Phase'], groups=METRIC_GROUPS, writer=writer, prefix='ablation_province', sort_cols=['Phase', 'RMSE'])
    print(f'Grouped metric tables saved to: {out_path}')
if VIS_SWITCH['data_overview']:
    vis_try('data_overview', make_data_overview_plots)
if VIS_SWITCH['feature_selection']:
    vis_try('feature_selection', make_feature_selection_plots)
if VIS_SWITCH['graph_structure'] or FORCE_ADJ_VIS:
    vis_try('graph_structure_mandatory', make_graph_structure_plots)
if VIS_SWITCH['ceemdan_branch']:
    vis_try('ceemdan_branch', make_ceemdan_branch_plots)
if VIS_SWITCH['prediction_fit']:
    vis_try('prediction_fit', make_prediction_fit_plots)
if VIS_SWITCH['model_metrics']:
    vis_try('model_metrics', make_model_metric_plots)
if VIS_SWITCH['ablation']:
    vis_try('ablation', make_ablation_plots)
if VIS_SWITCH['fusion']:
    vis_try('fusion', make_fusion_plots)
if VIS_SWITCH['consistency']:
    vis_try('consistency', make_consistency_plots)
if VIS_SWITCH['grouped_metric_excel']:
    vis_try('grouped_metric_excel', export_grouped_metric_excel)
print(f'\nAll visualizations have been saved to: {VIS_DIR}\n')
